# Plotting notebook for PiC_UVnudge runs
## Set up
### Packages

In [1]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
import scipy
from scipy import stats
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.mathtext import _mathtext as mathtext
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib import gridspec, animation
import matplotlib.path as mpath
import matplotlib.colors as colors
import matplotlib.dates as mdates
from matplotlib.patches import Ellipse
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from cmcrameri import cm
import jinja2
from Plotting_functions import wvl2wvn, wvn2wvl, p2z, z2p, t_test_two_means, Wilks_pcrit, CustomCmap, draw_circle
import cftime
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from functools import partial
from collections import defaultdict
from pathlib import Path
import itertools
import sys
sys.path.insert(0, "../processing_scripts")
from Processing_functions import lats, arclats, lons
import string

In [2]:
font_path = '/glade/work/glydia/conda-envs/Helvetica.ttf' 
boldfont_path = '/glade/work/glydia/conda-envs/Helvetica-Bold.ttf' 
font_manager.fontManager.addfont(font_path)
font_manager.fontManager.addfont(boldfont_path)

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

### Filepaths and name variables

In [3]:
## Plot types to make - CHANGE
# 0: Weighted spatial mean or Arctic sea ice area
# 1: Spatial maximum (AMOC)
# 2: Leave alone (doing spatial map or sea ice concentration)
plots = {
    'amoc': [False, 1], # Plot AMOC
    'lin': [False, 0],  # Plot non-AMOC linear
    'map': [False, 2],  # Plot spatial average map
    'mtrd': [False, 0],  # Plot monthly and annual trends
    'strd': [False, 2], # Plot spatial trends
    'zon': [False, 3], # Plot zonal data
    'ztrd': [False, 3], # Plot zonal trend data
    'comb': [True, 0], # Plot linear and monthly/annual trends together 
}

## Categorical plot type - DO NOT CHANGE
plot_types = {
    'spatial': False,
    'line': False,
    'mtrd': False,
    'zonal': False
}

# Set up plot_types based on plots
for pl, att in plots.items():
    if att[0]:
        if pl == 'mtrd':
            plot_types['mtrd'] = True
        elif att[1] <= 1:
            plot_types['line'] = True
        elif att[1] == 2:
            plot_types['spatial'] = True
        elif att[1] == 3:
            plot_types['zonal'] = True

# Spatial & time domain - CHANGE s_domain & t_domain only
s_domain = 1 # 0: Global, 1: Arctic, 2: Mid-latitudies
r_domain = False # True: use Arctic regional domain, False: pan-Arctic average
a_domain = plot_types['spatial'] or plot_types['zonal'] # True: 50-90, False: 70-90
o_domain =  True # True: 20-60N max at each lat, False: 33-55N max
subset_time = False # True: slice time in MasterDS, False: Need all data post-1950
t_domain = 1950 # start year
trd_length = 20 # Length of running trends - used for mtrd plots only
remove_amoc = False # Remove influence of AMOC

## Time averaging type - CHANGE
time_avg = 4   # 0: Monthly, 1: Yearly, 2: Seasonal, 3: All data, 4: Timeseries

## Ensemble mean or All members - CHANGE
ens_type = 0  # 0: All_members, 1: Mean

## Variables - CHANGE
# Primary variable
comp = 'ice'
freq = 0 # 0: monthly, 1: daily
var_ind = 0

# Secondary variable
comp2 = 'ocn'
var_ind2 = 0

# CHANGE
type_onemonth = 9 # Selects single month to plot timeseries for (time_avg = 4), if doing all months, equals 0
ocn_lat = 26.5
trend_months = [3,6,9,12]
plot_levels = [300,500, 850,925] # Selects plot levels for vlev type plots

In [4]:
# DO NOT CHANGE
var_list = {'atm': ['TREFHT','PSL','RESTOM','Z3','FSNTOA','TGCLDLWP','AEROD_v'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]
var2 = var_list[comp2][var_ind2]+var_ext[freq]

In [5]:
## Test numbers - DO NOT CHANGE
tst_nums = np.arange(1,4)

## Test names - True means plot, False means don't plot
# CHANGE
ds_plot_list = {
    'PiC_UVnudge_LM2006': True,
    'HIST_UVnudge_LM': True,
    'GHG_UVnudge_LM': True,
    'LENS2 piControl': True,
    'CESM2-lessmelt HIST': False,
    'LENS2 HIST-SSP370': True,
    'GHG2': True,
    # 'GISTEMP': False,
    # 'HadCRUT5': False,
    # 'BEST': False,
    'DA_SIC_RECON': False,
    # 'NSIDC': False,
    # 'OSISAF': False,
    # 'ERA5': False,
    # 'RAPID': False,
    # 'CERES': False
}

## Filepaths - DO NOT CHANGE
path_to_work = '/glade/work/glydia/'
path_to_data = path_to_work+'Arctic_controls_processed_data/recent_plotting_data/'
path_to_graphs = '/glade/u/home/glydia/Recent_PiC_HIST_nudge_graphs/'

# Conditions - DO NOT CHANGE
vert_lev = {'atm': [False,False,False,True,False,False,False],
            'ice': [False,False],
            'ocn': [False]}
file_bool = not vert_lev[comp][var_ind] and freq == 0

In [6]:
########################## DO NOT CHANGE ANYTHING BELOW THIS LINE #############################

In [7]:
%%time
    
## Select plot type
time_str_list = {0: 'month', 1: 'year', 2: 'season', 3: 'all', 4: 'timeseries'}
time_outstr = time_str_list[time_avg]

## Select ensemble type
ens_str_list = {0: 'All_members', 1: 'Mean'}
ens_str = ens_str_list[ens_type]

## Select time and spatial domain strings
sd_str_list = {0: 'Global', 1: 'Arctic', 2: 'MidLat'}
s_domain = 0 if (var == 'MOC') else s_domain # Make sure MOC is global
sd_str = sd_str_list[s_domain]

rd_str_list = {True: 'Reg', False: ''}
rd_str = rd_str_list[r_domain]

od_str_list = {True: 'Lat', False: ''}
od_str = od_str_list[o_domain]

td_str = str(t_domain)
t_end = 2024

month_str = np.array(['January','February','March','April','May','June','July','August','September','October','November','December'])
mon_str = np.array([i[0:3] for i in month_str])
m_str = np.array([i[0] for i in month_str])
seas_str = np.array(['MAM','JJA','SON','DJF'])
mon_seas_dict = {3: 'MAM',
                 6: 'JJA',
                 9: 'SON',
                12: 'DJF'}

CPU times: user 65 µs, sys: 0 ns, total: 65 µs
Wall time: 71 µs


In [8]:
## Set up properties of each dataset and whether it will be plotted or not
# O: LENS ensemble
# 1: PiC_UVnudge ensemble
# 2: PiC_UVnudge single run
# 3: observations
# attribute structure: [use dataset[0], dataset typ[1], line color[2], line style[3], zorder[4], note[5](optional)]
use_era5 = var in ['aice','TREFHT','Z3']
use_ceres = var in ['RESTOM','FSNTOA','TOT_CLD_VISTAU']
ds_names = {
    'ERA5': [use_era5, 3, 'black', '-', 10, 'era5'],
    'GISTEMP': [var == 'TREFHT' and (plot_types['mtrd'] or plot_types['line']), 3, 'black', '-', 10 , 'gist'],
    'HadCRUT5': [var == 'TREFHT' and (plot_types['mtrd'] or plots['comb'][0]), 3, 'black', '-', 10 , 'had5'],
    'BEST': [var == 'TREFHT' and (plot_types['mtrd'] or plots['comb'][0]), 3, 'black', '-', 10 , 'best'],
    'DA_SIC_RECON': [var == 'aice' and (plot_types['mtrd'] or plot_types['line']), 3, 'black', '-', 10, 'dasr'],
    'NSIDC': [var == 'aice' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'nsii'],
    'OSISAF': [var == 'aice' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'osii'],
    'RAPID': [var == 'MOC' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'rap'],
    'CERES': [use_ceres, 3, 'black', '-', 10, 'ceres'],
    'LENS2 piControl': [True, 0, 'tab:blue', '-', 1],
    'CESM2-lessmelt HIST': [True, 0, 'tab:purple', '-', 1],
    'LENS2 HIST-SSP370': [True, 0, 'tab:red', '-', 1],
    'GHG2': [True, 0, 'tab:green', '-', 1],
    'PiC_UVnudge_LM2006': [None, 1, 'blue', '-', 4, 'lm06'],
    'HIST_UVnudge_LM': [None, 1, 'red', '-', 5, 'lmh'],
    'GHG_UVnudge_LM': [None, 1, 'lime', '-', 5, 'lmghg'],
}

ds_attr_names = {'Non-GHG': [2, 'tab:orange'],
             'Greenhouse gases': [1, 'tab:green'],
             'Winds': [3, 'tab:blue'],
             'HIST+winds': [0, 'tab:red'],
             'HIST-control': [0, 'tab:red'],
             'Non-GHG-control': [2, 'tab:orange'],
             'Greenhouse gases-control': [1, 'tab:green'],}

# ds_2order_names = 

ds_paper_names = {
    'CESM2-lessmelt HIST': 'HIST-lessmelt-control',
    'LENS2 HIST-SSP370': 'HIST-SSP370-control',
    'GHG2': 'GHG-control',
    'LENS2 piControl': 'PI-control',
    'PiC_UVnudge_LM2006': 'PI+winds',
    'ERA5': 'OBS (ERA5)',
    'GISTEMP': 'OBS (GISTEMP)',
    'HadCRUT5': 'OBS (HadCRUT5)',
    'BEST': 'OBS (BEST)',
    'NSIDC': 'OBS (NSIDC)',
    'OSISAF': 'OBS (OSISAF)',
    'DA_SIC_RECON': 'OBS (DA SIC)',
    'RAPID': 'OBS (RAPID)',
    'CERES': 'OBS (CERES)',
    'HIST_UVnudge_LM': 'HIST+winds',
    'GHG_UVnudge_LM': 'GHG+winds',
}

region_names = {
    'HA': 'High Arctic',
    'G': 'Greenland',
    'AA': 'Atlantic Arctic',
    'WS': 'Western Siberia',
    'ES': 'Eastern Siberia',
    'PA': 'Pacific Arctic'
}

# Update PiC_UVnudge 'use dataset' values with T/F from ds_plot_list
for dsname, use in ds_plot_list.items():
    ds_names[dsname][0] = use
    if dsname == 'ERA5':
        use_era5 = use

In [9]:
## Determine note type based on model runs plotted
note = ''
driftnote = ''

for dsname, attrs in ds_names.items():
    # Only pick datasets that will plotted and that have a code
    if attrs[0] and len(attrs) == 6:
        # Each PiC_UVnudge run has its own code, if the run will be plotted, it's code will be added to the note
        note = note+attrs[5]+'-'

note = 'era5' if note == '' else note[:-1]
driftnote = driftnote if driftnote == '' else driftnote[:-1]

### Custom functions
#### LoadData

In [10]:
def LoadData(plot_type, varname, tavg, plot_level=None):
    # Create file name     
    sd_i = 'Global' if varname == 'MOC' else sd_str
    domain_str = sd_i
    if r_domain:
        domain_str += '.'+rd_str
    if o_domain:
        domain_str += '.'+od_str
    filename = plot_type+'.'+varname+'.'+domain_str+'.'+td_str+'.'+tavg+'.'+ens_str+'.nc'
    filename2 = plot_type+'.'+varname+'.'+domain_str+'.'+td_str+'.'+tavg+'.All_members.nc'
    filename3 = plot_type+'.'+varname+'.'+sd_i+'.'+'1950'+'.'+tavg+'.'+ens_str+'.nc'
    filepath = path_to_data+filename

    # If file exists, open
    if Path(filepath).exists():
        print(filename)
        data = xr.open_dataset(filepath)

    # If ensemble mean and version with all members exists, open and take ensemble mean
    elif ens_type and Path(path_to_data+filename2).exists():
        print(filename2)
        data = xr.open_dataset(path_to_data+filename2)
        data = data.mean('ensemble_member')

    # If ensemble mean and version starting in 1950 exists, open
    elif ens_type and Path(path_to_data+filename3).exists():
        print(filename3)
        data = xr.open_dataset(path_to_data+filename3)
    
    # Else, requested ensemble or time domain type doesn't exist but version with all members and starting in 1950 does
    else:
        filename = plot_type+'.'+varname+'.'+sd_i+'.'+'1950'+'.'+tavg+'.All_members.nc'
        filepath = path_to_data+filename
        print(filename)
        data = xr.open_dataset(filepath)
        if ens_type:
            data = data.mean('ensemble_member')
    
    return data

#### SubCmap

In [11]:
def SubCmap(orgcmap, levels, type, color):
    # Type is middle, end or beginning
    # Make cmap values swapped out
    # color is color to sub in
    lev_len = len(levels)+1
    cmap_samp = orgcmap.resampled(lev_len)
    cmap_list = cmap_samp(np.linspace(0,1,lev_len))

    # Swap out first two
    if type == 'beg':
        lev_ind = 0
        cmap_list[lev_ind:(lev_ind+1), :] = color[0]
        cmap_list[(lev_ind+1):(lev_ind+2), :] = color[1]
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [color[0], orgcmap.get_over()], True)

    # Swap out middle two
    elif type == 'mid':
        lev_ind = int(lev_len/2)
        cmap_list[(lev_ind-1):(lev_ind+1), :] = color
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [orgcmap.get_under(), orgcmap.get_over()], True)

    # Swap out last
    else:
        lev_ind = -1
        cmap_list[lev_ind:, :] = color
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [orgcmap.get_under(), color], True)

    return final_cmap

#### TrendLine

In [12]:
def TrendLine(da, period):
    da = da.dropna(dim=period)
    # Calculate regression coefficients
    da_reg_coef = da.polyfit(dim=period, deg=1)

    # Calculate trend line
    da_yreg = (da_reg_coef.loc[dict(degree=1)]*da[period])+da_reg_coef.loc[dict(degree=0)]

    return da_yreg['polyfit_coefficients']

#### DetrendAMOC

In [13]:
def DetrendAMOC(ds, ds2, ty, v):
    # ds: aice or TREFHT, ds2: AMOC, ty: var 1 annual (1) or monthly (0) variable
    print('Detrending '+v+' with '+var2)
    control_name = 'LENS2 piControl'
    if ty == 1:
        # Remove nans
        ds2_nan = ds2[control_name].values.flatten()
        ds2_nan = ds2_nan[~np.isnan(ds2_nan)]

        ds_nan = ds[control_name].values.flatten()
        ds_nan = ds_nan[~np.isnan(ds_nan)]
        
        # Create linear regression
        m, b, r, p, se = stats.linregress(ds2_nan, ds_nan)
        print('   slope: {:.3f}'.format(m))
        print('   inter: {:.3f}'.format(b))
        print('   r-val: {:.3f}'.format(r))
        print('   p-val: {:.3f}'.format(p))

        # Calculate effect of AMOC in sea ice and temperature
        ds_amoc_eff = m*ds2
        if v == 'aice':
            ds_amoc_eff = ds_amoc_eff.rename({'year':'time'})
            ds_amoc_eff = ds_amoc_eff.assign_coords({'time':ds.time})

        # Remove effect of AMOC
        ds_wo_amoc = ds-ds_amoc_eff

        # Add observed variables back in
        for dsname, attrs in ds_names.items():
            if attrs[1] == 3 and attrs[0] and dsname in ds:
                ds_wo_amoc[dsname] = ds[dsname]
    else:
        ds_list = []
        ds2_list = []
        # Remove nans
        ds2_nan = ds2[control_name].values.flatten()
        ds2_nan = ds2_nan[~np.isnan(ds2_nan)]
        for m in np.arange(1,13):
            ds_m = ds.loc[{'month':m}]
            print(month_str[m-1])

            # Remove nans
            ds_m_nan = ds_m[control_name].values.flatten()
            ds_m_nan = ds_m_nan[~np.isnan(ds_m_nan)]
            
            # Create linear regression
            m, b, r, p, se = stats.linregress(ds2_nan, ds_m_nan)
            print('   slope: {:.3f}'.format(m))
            print('   inter: {:.3f}'.format(b))
            print('   r-val: {:.3f}'.format(r))
            print('   p-val: {:.3f}'.format(p))
    
            # Calculate effect of AMOC in sea ice and temperature
            ds_m_amoc_eff = m*ds2
    
            # Remove effect of AMOC
            ds_m_wo_amoc = ds_m-ds_m_amoc_eff

            # Add observed variables back in
            for dsname, attrs in ds_names.items():
                if attrs[1] == 3 and attrs[0] and dsname in ds:
                    ds_m_wo_amoc[dsname] = ds_m[dsname]
                    
            ds_list.append(ds_m_wo_amoc)
            ds2_list.append(ds_m_amoc_eff)

        ds_wo_amoc = xr.concat(ds_list, dim=pd.Index(np.arange(1,13), name='month'))
        ds_amoc_eff = xr.concat(ds2_list, dim=pd.Index(np.arange(1,13), name='month'))
    
    return ds_wo_amoc, ds_amoc_eff

#### AttrCalc

In [14]:
def AttrCalc(ds, varname, skip_obs):
    use_obs = varname != 'MOC'  and varname != 'AEROD_v' and not skip_obs
    if use_obs:
        obs_name_list = [name for name, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]

    funcs = {0: np.subtract,
            1: np.add}

    # 0: subtract, 1: addition
    attr_names = {'Non-GHG': ['HIST_UVnudge_LM',0,'GHG_UVnudge_LM'],
                 'Anthropogenic forcing': ['HIST_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Greenhouse gases': ['GHG_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Winds': ['PiC_UVnudge_LM2006'],
                 'HIST+winds': ['HIST_UVnudge_LM'],
                 'HIST-control': ['LENS2 HIST-SSP370'],
                 'Greenhouse gases-control': ['GHG2'],
                 'Non-GHG-control': ['LENS2 HIST-SSP370',0,'GHG2']}
    if use_obs:
        for obs_name in obs_name_list:
            attr_names[ds_paper_names[obs_name]] = [obs_name]
            attr_names['Bias ('+ds_paper_names[obs_name][5:-1]+')'] = [obs_name,0,'HIST_UVnudge_LM']
    
    attr_list = []
    
    for aname, eq in attr_names.items():
        print(aname)
        # If term composed of one variable - no calculation needed
        if len(eq) == 1 and ds_names[eq[0]][0]:
            eqname = eq[0]+' mean' if ds_names[eq[0]][1] == 0 else eq[0]
            attr_list.append(ds[eqname].rename(aname))
            print('  Calculated')

        # If calculation needs two terms
        elif len(eq) > 1:
            # Find all variable names in calculation
            calc_names = eq[::2]

            # See if all names in calculation are set to be plotted, if not don't calculate this attribution term
            all = True
            for cname in calc_names:
                if not ds_names[cname][0]:
                    all = False
                    break
            if not all:
                continue
            else:
                eqname = eq[0] + ' mean' if ds_names[eq[0]][1] == 0 else eq[0]
                da_attr = ds[eqname]
                for i in np.arange(1,len(eq)-1,2):
                    eqname = eq[i+1] + ' mean' if ds_names[eq[i+1]][1] == 0 else eq[i+1]
                    da_attr = xr.apply_ufunc(funcs[eq[i]],da_attr,ds[eqname])
                attr_list.append(da_attr.rename(aname))
                print('  Calculated')

    ds_attr = xr.merge(attr_list)
    return ds_attr

#### CalcR

In [15]:
def CalcR(ds_attr):
    da_hist = ds_attr['HIST+winds']
    da_hist_short = da_hist.loc[dict(start_year=slice(1980,2005))]
    corr_dict = {}

    for aname, da in ds_attr.items():
        if 'OBS' in aname:
            if 'NSIDC' in aname or 'OSISAF' in aname:
                corr = xr.corr(da_hist_short, da.loc[dict(start_year=slice(1980,2005))])
            else:
                corr = xr.corr(da_hist, da)
            corr_dict[aname] = corr.values
    return corr_dict  

#### AddMidYear

In [16]:
def AddMidYear(ds_trd):
    ds_trd = ds_trd.assign_coords(mid_year=('start_year',ds_trd.start_year.values+10))
    return ds_trd

#### CalcPIStatSig

In [17]:
def CalcPIStatSig(da_pi, ds_attr):
    # Calculate pre-industrial statistics
    pimean = da_pi.mean('slice', skipna=True)
    pistd = da_pi.std('slice', skipna=True)

    # Z statistic
    zstat = (ds_attr-pimean)/pistd

    # P-value
    dz_list = []
    for daname, dz in zstat.items():
        pval = stats.norm.sf(abs(dz.values))**2
        dpval = dz.copy(data=pval)
        dz_list.append(dpval)
    
    ds_pval = xr.merge(dz_list)
    return ds_pval

#### LoadRegridder

In [18]:
def LoadRegridder():
    filestem = 'ERAregridder'
    method = 'bilinear'
    
    # Check for existing target grid
    targetgridpath = path_to_data+filestem+'.'+sd_str+'.targetgrid.nc'

    # Check for existing sample grid
    samplegridpath = path_to_data+filestem+'.'+sd_str+'.samplegrid.nc'

    # Check for existing regridder weights
    weightpath = path_to_data+filestem+'.'+sd_str+'.weights.nc'

    # If exists, open file for target grid
    target_grid = xr.open_dataset(targetgridpath)

    # If exists, open file
    sample_grid = xr.open_dataset(samplegridpath)
   
    regridder = xe.Regridder(sample_grid, target_grid, method, filename=weightpath,reuse_weights=True)

    return regridder

#### RegridERA5

In [19]:
def RegridERA5(ds, regridder):
    ## Regridding
    print('Regridding ERA5 grid -> ATM grid...')
    da = ds.rename({'latE': 'lat', 'lonE': 'lon'})
    da = da.where(da.lon != 180.0, drop=True)
    da_re = regridder(da)
    da_re = da_re.rename({'x':'lon', 'y':'lat'})
    da_re = da_re.assign_coords({'lon':lons, 'lat': arclats})
    da_re = da_re.rename('ERA5')
    

    # Adding back in cyclic point
    cyclic_data, cyclic_lon = add_cyclic_point(da_re.data, coord=da_re['lon'])
    cyclic_coords = {dim: da_re.coords[dim] for dim in da_re.dims}
    cyclic_coords['lon'] = cyclic_lon
    da_re = xr.DataArray(cyclic_data, dims=da_re.dims, coords=cyclic_coords, attrs=da_re.attrs, name=da_re.name)
    
    return da_re

#### FigLabelList

In [20]:
def FigLabelList(total):
    # total = total number of letters needed for subfigure label list
    lowercase = string.ascii_lowercase
    label_list = ['('+lowercase[i]+')' for i in range(0,total)]
    return label_list

#### AxisLabels

In [21]:
def AxisLabels(ax, title_str, land):
    if land:
        ax.add_feature(cfeature.LAND, zorder=3)
    ax.coastlines(zorder=4)
    ax.set_extent(extent, ccrs.PlateCarree())
    ax.set_title(title_str, fontsize = 10)
    draw_circle(ax, draw_circ=(s_domain == 1), draw_major=False)
    return None

#### SaveFig

In [22]:
def SaveFig(fig, plot_type, tavg, varname, note=None, plot_level=None):
    # File format is:
    # var.ensemble_type.spatialdomain.[Zplot_level].plot_type.timeaveraging.timedomain.[note].png
    level_str = '' if plot_level == None else 'Z'+str(plot_level)+'.'
    note_str = '' if note == None else note+'.'
    
    fig.savefig(path_to_graphs+varname+'.'+ens_str+'.'+sd_str+'.'+level_str+plot_type+'.'+tavg+'.'+td_str+'.'+note_str+'pdf', bbox_inches='tight')
    return None

## Month trend plots
### Set up

In [23]:
if plot_types['mtrd']:
    graph_type_str = 'Linear.Trend' 

    # Graph labels and variables
    xtick_loc = np.arange(1,13)
    xtick_lbl = mon_str
    xlim = [0.9,12.1]
    xlabel= 'Month'
    date_str = 'Month'
    period = 'month'
    dim_avg = 'time.month'
        

    ylabel = {'aice': 'Sea ice extent trend (million km$^2$/decade)',
             'TREFHT': '2m air temperature trend (K/decade)',
             'AEROD_v': 'Aerosol optical depth trend (/decade)',
             }
    ylabel2line = {'aice': 'Sea ice extent trend\n(million km$^2$/decade)',
             'TREFHT': '2m air temperature\ntrend (K/decade)',
             'AEROD_v': 'Aerosol optical depth\ntrend (/decade)'}
    ylabelrat = {'aice': 'Percent of OBS sea ice extent\ntrend explained (%)',
              'TREFHT': 'Percent of OBS 2m air temperature\ntrend explained (%)'}
    yticks = {'aice': np.arange(-1.2, 0.41,0.2),
              'TREFHT': np.arange(-1.0,1.51,0.5)}
    yticksrat = {'aice': np.arange(-10,51,10),
                 'TREFHT': np.arange(-20,51,10)}
    ylim = {'aice': [-1.3,0.4],
            'TREFHT': [-1,2]}
    ylimrat = {'aice': [-15,50],
            'TREFHT': [-20,60]}
    bbox_loc = {'aice':(0.01, 0.45),
              'TREFHT': (0.21, 1.03)}
    bbox_loc_rat = {'aice':(0.25, 0.99),
                    'TREFHT': (0.45, 0.99)}

    if not subset_time:
        trdyr_str = '.'+str(trd_length)+'yr'
        ramoc_str = '.DtrdAMOC' if remove_amoc else ''
        if o_domain:
            ramoc_str += '.'+str(ocn_lat)+'N'
        else:
            ramoc_str += '.33-55N'
        
        mcmaps = {'aice': cm.vik_r,
                 'TREFHT': cm.vik,
                 'AEROD_v': cm.vik,
                 'MOC': cm.vik}
        mdcmaps = {'aice': cm.cork,
                  'TREFHT': cm.cork,
                  'AEROD_v': cm.cork}
        mlevels = {'aice': np.arange(-1.80,1.81,0.15),
                  'TREFHT': np.arange(-3,3.1,0.25),
                  'AEROD_v': np.array([-0.05,-0.04,-0.03,-0.02,-0.01,-0.009,-0.008,-0.007,-0.006,-0.005,-0.004,-0.003,-0.002,-0.001,0.000,
                                      0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008,0.009,0.01,0.02,0.03,0.04,0.05]),
                  'MOC': np.arange(-4,4.01,0.25)}
        mticks = {'aice': np.arange(-1.8,1.81,0.6),
                 'TREFHT': np.arange(-3,3.1,1.0),
                 'AEROD_v': np.arange(-0.05,0.051,0.01),
                 'MOC': np.arange(-4,4.01,1.0)}
        mlabels = {'aice': np.arange(-1.5,1.6,0.5),
                   'TREFHT': np.arange(-2,2.1,1.0)}
        xlim = [t_domain+10,t_end-(trd_length-1)+10]
        ylim = {'aice': [-2,1],
               'TREFHT': [-2,2],
               'MOC': [-4,4],
               'AEROD_v': [-0.055,0.055]}
        ylimcontr = {'aice': [-2.2,1.2],
                    'TREFHT': [-1.2,1.8],
                    'MOC': [-5,5],
                    'AEROD_v': [-0.055,0.055]}
        ywidth = ((t_end-(trd_length-1))-t_domain)*(5/55)
        ylabelsingle = {'aice': mon_str[type_onemonth-1]+' sea ice extent trend (million km$^2$/decade)',
                        'TREFHT': 'Annual 2m air temperature trend (K/decade)',
                       'MOC': 'Annual Atlantic Meridional Overturning Circulation trend (Sv/decade)',
                       'AEROD_v': mon_str[type_onemonth-1]+' aerosol optical depth trend (/decade)'}
        ylabeltwolinesingle = {'aice': mon_str[type_onemonth-1]+' sea ice extent\ntrend (million km$^2$/decade)',
                        'TREFHT': 'Annual 2m air temperature\ntrend (K/decade)',
                       'MOC': 'Atlantic Meridional Overturning\nCirculation trend (Sv/decade)',
                        'AEROD_v': mon_str[type_onemonth-1]+' aerosol optical depth\ntrend (/decade)'}
        xlabel = 'Mid year of '+str(trd_length)+'-year trend'
        xdim = 'mid_year'
        bbox_loc = {'aice': 'lower left',
                    'TREFHT': 'lower right',
                    'MOC': 'lower left',
                    'AEROD_v': 'best'}

### Data loading

In [24]:
%%time

if plot_types['mtrd']:
    
    if subset_time:
        # Load monthly trends
        ds_mon_trd = LoadData(graph_type_str, var, 'month')
    
        # Load annual trends
        ds_ann_trd = LoadData(graph_type_str, var, 'year')

        ds_attr_mon = AttrCalc(ds_mon_trd, var, False)
        ds_attr_ann = AttrCalc(ds_ann_trd, var, False)

    else:
        trdyr_str = str(trd_length)+'yr'
        if var != 'MOC':
            # Load monthly trends
            ds_mon_trd = LoadData(graph_type_str+'.'+trdyr_str, var, 'month')
            ds_mon_trd = AddMidYear(ds_mon_trd)

        # Load trends
        ds_ann_trd = LoadData(graph_type_str+'.'+trdyr_str, var, 'year')
        ds_ann_trd = AddMidYear(ds_ann_trd)

        if var != 'MOC':
            obs_name_list = [ds_paper_names[name] for name, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]
            obs_len = len(obs_name_list)
            for i in range(0,obs_len*2,2):
                obs_name = obs_name_list[int(i/2)]
                ds_attr_names[obs_name] = [i+2, 'k']
                ds_attr_names['Bias ('+obs_name[5:-1]+')'] = [i+3, 'orchid']
                
            if var != 'aice':
                ds_attr_ann = AttrCalc(ds_ann_trd, var, False)
                ds_pval_ann = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann)
            ds_attr_mon = AttrCalc(ds_mon_trd, var, False)
            ds_pval_mon = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_attr_mon)
        else:
            ds_attr_ann = AttrCalc(ds_ann_trd, var, False)
            ds_pval_ann = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann)

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 6.91 µs


In [25]:
%%time

# Secondary variable
if plot_types['mtrd'] and not subset_time:
    # Load secondary variable - MOC
    # Load annual trends
    ds2_ann_trd = LoadData(graph_type_str+'.'+trdyr_str, var2, 'year')   
    ds2_ann_trd = AddMidYear(ds2_ann_trd)
    ds2_ann_trd = ds2_ann_trd.sel(lat=ocn_lat, method='nearest')

    ds2_attr_ann = AttrCalc(ds2_ann_trd, var2, False)
    ds2_pval_ann = CalcPIStatSig(ds2_ann_trd['LENS2 piControl'], ds2_attr_ann)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.96 µs


In [26]:
%%time

# Remove effect of AMOC
if plot_types['mtrd'] and not subset_time and remove_amoc:
    # Remove AMOC 
    if var != 'aice':
        ds_ann_trd_amoc, ds_ann_trd_amoc_eff = DetrendAMOC(ds_ann_trd, ds2_ann_trd, 1, var)
        ds_attr_ann_amoc = AttrCalc(ds_ann_trd_amoc, var, False)
        ds_pval_ann_amoc = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann_amoc)
        ds_pval_ann_amoc_eff = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_ann_trd_amoc_eff)
    
    ds_mon_trd_amoc, ds_mon_trd_amoc_eff = DetrendAMOC(ds_mon_trd, ds2_ann_trd, 0, var)
    ds_attr_mon_amoc = AttrCalc(ds_mon_trd_amoc, var, False)
    ds_pval_mon_amoc = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_attr_mon_amoc)
    ds_pval_mon_amoc_eff = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_mon_trd_amoc_eff)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.72 µs


In [27]:
if plot_types['mtrd']  and var != 'MOC':
    patt_corr_mon = CalcR(ds_attr_mon)
    
    print('Pattern correlations (with AMOC)')
    for name, corr in patt_corr_mon.items():
        print('  '+name+': '+format(corr, '.2f'))

    if remove_amoc:
        patt_corr_mon_amoc = CalcR(ds_attr_mon_amoc)
    
        print('Pattern correlations (without AMOC)')
        for name, corr in patt_corr_mon_amoc.items():
            print('  '+name+': '+format(corr, '.2f'))

In [28]:
# Set variables according to AMOC removal status
if plot_types['mtrd'] and var != 'MOC':
    if remove_amoc:
        if var != 'aice':
            dsM_ann_trd = ds_ann_trd_amoc
            dsM_attr_ann = ds_attr_ann_amoc
            dsM_pval_ann = ds_pval_ann_amoc
        dsM_mon_trd = ds_mon_trd_amoc
        dsM_attr_mon = ds_attr_mon_amoc
        dsM_pval_mon = ds_pval_mon_amoc
        Mpatt_corr_mon = patt_corr_mon_amoc
        
    elif not remove_amoc:
        if var != 'aice':
            dsM_ann_trd = ds_ann_trd
            dsM_attr_ann = ds_attr_ann
            dsM_pval_ann = ds_pval_ann
        dsM_mon_trd = ds_mon_trd
        dsM_attr_mon = ds_attr_mon
        dsM_pval_mon = ds_pval_mon
        Mpatt_corr_mon = patt_corr_mon

### Make plots
#### Subsetted time

In [29]:
%%time

if plot_types['mtrd'] and subset_time:
    ## Make standard monthly trend plots
    # Set up 
    fig = plt.figure(figsize=(4,3.5), layout='constrained')
    gs = fig.add_gridspec(1, 2,  width_ratios=(6, 1), wspace=0)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Plotting
    ax1.axhline(0,color='k',lw=0.5,zorder=0)
    ax2.axhline(0,color='k',lw=0.5,zorder=0)
    
    for dsname, attrs in ds_names.items():
        if attrs[0]:
            da_mon = ds_mon_trd[dsname]
            da_ann = ds_ann_trd[dsname]

            # LENS ensemble
            if attrs[1] == 0:
                # Extract mean & min and max of envelople
                da_mon_mean = ds_mon_trd[dsname+' mean']
                da_mon_min = ds_mon_trd[dsname+' min']
                da_mon_max = ds_mon_trd[dsname+' max']

                da_ann_mean = ds_ann_trd[dsname+' mean']
                da_ann_min = ds_ann_trd[dsname+' min']
                da_ann_max = ds_ann_trd[dsname+' max']
                
                # Plot envelope
                ax1.fill_between(da_mon_min[period].values, da_mon_min.values,
                    da_mon_max.values, color=attrs[2], alpha=0.1,
                    ec=None, zorder=attrs[4],label=ds_paper_names[dsname])
                ax2.axhspan(ymin=da_ann_min.values, ymax=da_ann_max.values,
                   xmin=0.45, xmax=0.55, alpha=0.1, color=attrs[2],ec=None,zorder=attrs[4])
                
                # Plot mean
                # da_mon_mean.plot(
                #     ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4],label=ds_paper_names[dsname]+' mean')
                # ax2.axhline(da_ann_mean.values,xmin=0.45,xmax=0.55, ls=attrs[3],
                #     zorder=attrs[4], color=attrs[2])

            # PiC_UVnudge ensemble
            elif attrs[1] == 1:
                index = dict(ensemble_member=1)

                # Plot first ensemble member
                da_mon.loc[index].plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname])
                ax2.axhline(da_ann.loc[index].values, xmin=0.45, xmax=0.55, zorder=attrs[4], 
                            ls=attrs[3], color=attrs[2]) 

                # Plot other ensemble members
                for i in range(2,4):
                    index['ensemble_member'] = i
            
                    da_mon.loc[index].plot(
                        ax=ax1, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                    ax2.axhline(da_ann.loc[index].values, xmin=0.45, xmax=0.55,
                           zorder=attrs[4], color=attrs[2], ls=attrs[3])
                    
            # PiC_UVnudge single run & Observations
            elif attrs[1] >= 2:
                # Plot monthly & annual trends
                da_mon.plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname])
                ax2.axhline(da_ann.values, xmin=0.45, xmax=0.55,
                    color=attrs[2], ls=attrs[3], zorder=attrs[4])
    
    # Formatting
    ax1.set_title('')
    ax1.set_xlim(xlim)
    ax1.set_ylim(ylim[var])
    ax1.set_ylabel(ylabel[var])
    ax1.set_xticks(ticks=xtick_loc, labels=xtick_lbl, rotation='vertical')
    ax1.set_xlabel(xlabel)
    ax1.legend(loc='upper left', framealpha=0, bbox_to_anchor=bbox_loc[var], fontsize=8)
    ax1.grid(alpha=0.3)
    
    ax2.set_ylim(ylim[var])
    ax2.set_yticks([])
    ax2.set_xticks(ticks=[0.5],labels=['ANNUAL'],rotation='vertical')
    ax2.grid(alpha=0.3)
    for tick in yticks[var]:
        ax2.axhline(tick,color='tab:gray',zorder=0,lw=0.5,alpha=0.3)

    SaveFig(fig, graph_type_str, 'monyr', var, note)

    plt.close(fig)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.63 µs


In [30]:
%%time

if plot_types['mtrd'] and subset_time:
    frame = {'model': ['HIST+Winds','Greenhouse Gases','Non-GHG','Winds'],}
              # 'model+obs': ['HIST+Winds',ds_paper_names[obs_name],'Bias']} 
    # Setup figure
    for f, fvlist in frame.items():
        df_attr_mon = ds_attr_mon[fvlist]
        df_attr_ann = ds_attr_ann[fvlist]

        ## Make attribution monthly trend plots
        # Set up 
        fig = plt.figure(figsize=(4,3.5), layout='constrained')
        gs = fig.add_gridspec(1, 2,  width_ratios=(6, 1), wspace=0)
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
    
        # Plotting
        ax1.axhline(0,color='k',lw=0.5,zorder=0)
        ax2.axhline(0,color='k',lw=0.5,zorder=0)

        for daname,_ in df_attr_mon.items():
            da_attr_mon = df_attr_mon[daname]
            da_attr_ann = df_attr_ann[daname]
            attrs = ds_attr_names[daname]
            
            if ens_type:
                da_attr_mon.plot(
                        ax=ax1, color=attrs[1], x=period, label=daname)
                ax2.axhline(da_attr_ann.loc[index].values, xmin=0.45, xmax=0.55, color=attrs[1])
            else:
                da_attr_mon_mean = da_attr_mon.mean('ensemble_member')
                da_attr_mon_min = da_attr_mon.min('ensemble_member')
                da_attr_mon_max = da_attr_mon.max('ensemble_member')

                da_attr_ann_mean = da_attr_ann.mean('ensemble_member')
                da_attr_ann_min = da_attr_ann.min('ensemble_member')
                da_attr_ann_max = da_attr_ann.max('ensemble_member')

                # Monthly
                # Plot envelope
                ax1.fill_between(da_attr_mon_min[period].values, da_attr_mon_min.values,
                    da_attr_mon_max.values, color=attrs[1], alpha=0.2,
                    ec=None)
                
                # Plot mean
                da_attr_mon_mean.plot(
                    ax=ax1, color=attrs[1], x=period,label=daname)

                # Annual
                ax2.axhspan(ymin=da_attr_ann_min.values, ymax=da_attr_ann_max.values,
                   xmin=0.45, xmax=0.55, alpha=0.2, color=attrs[1],ec=None)

                ax2.axhline(da_attr_ann_mean.values,xmin=0.45,xmax=0.55,color=attrs[1])
           
        # Formatting
        ax1.set_title('')
        ax1.set_xlim(xlim)
        ax1.set_ylim(ylim[var])
        ax1.set_ylabel(ylabel[var])
        ax1.set_xticks(ticks=xtick_loc, labels=xtick_lbl, rotation='vertical')
        ax1.set_xlabel(xlabel)
        ax1.legend(loc='upper left', framealpha=0, bbox_to_anchor=bbox_loc[var], fontsize=8)
        ax1.grid(alpha=0.3)
        
        ax2.set_ylim(ylim[var])
        ax2.set_yticks([])
        ax2.set_xticks(ticks=[0.5],labels=['ANNUAL'],rotation='vertical')
        ax2.grid(alpha=0.3)
        for tick in yticks[var]:
            ax2.axhline(tick,color='tab:gray',zorder=0,lw=0.5,alpha=0.3)
    
        SaveFig(fig, graph_type_str, 'monyr-'+f, var, note)
    
        plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.44 µs


In [31]:
%%time

if plot_types['mtrd'] and subset_time:
    # Make ratio of exp/obs trend explained plots
    # Set up 
    fig = plt.figure(figsize=(4,3.5), layout='constrained')
    gs = fig.add_gridspec(1, 2,  width_ratios=(6, 1), wspace=0)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Plotting
    ax1.axhline(0,color='k',lw=0.5,zorder=0)
    ax2.axhline(0,color='k',lw=0.5,zorder=0)

    da_mon_era = ds_mon_trd['ERA5']
    da_ann_era = ds_ann_trd['ERA5']
    
    for dsname, attrs in ds_names.items():
        if attrs[0]:
            da_mon = ds_mon_trd[dsname]
            da_ann = ds_ann_trd[dsname]

            # PiC_UVnudge ensemble
            if attrs[1] == 1:
                index = dict(ensemble_member=1)

                # Plot first ensemble member
                ((da_mon.loc[index]/da_mon_era)*100).plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+'/\n'+ds_paper_names['ERA5'])
                ax2.axhline(((da_ann.loc[index]/da_ann_era)*100).values, xmin=0.45, xmax=0.55, zorder=attrs[4], 
                            ls=attrs[3], color=attrs[2]) 

                # Plot other ensemble members
                for i in range(2,4):
                    index['ensemble_member'] = i
            
                    ((da_mon.loc[index]/da_mon_era)*100).plot(
                        ax=ax1, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                    ax2.axhline(((da_ann.loc[index]/da_ann_era)*100).values, xmin=0.45, xmax=0.55,
                           zorder=attrs[4], color=attrs[2], ls=attrs[3])
                    
            # PiC_UVnudge single run & Observations
            elif attrs[1] == 2:
                # Plot monthly & annual trends
                ((da_mon/da_mon_era)*100).plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+'/\n'+ds_paper_names['ERA5'])
                ax2.axhline(((da_ann/da_ann_era)*100).values, xmin=0.45, xmax=0.55,
                    color=attrs[2], ls=attrs[3], zorder=attrs[4])
    
    # Formatting
    ax1.set_title('')
    ax1.set_xlim(xlim)
    ax1.set_ylim(ylimrat[var])
    ax1.set_ylabel(ylabelrat[var])
    ax1.set_xticks(ticks=xtick_loc, labels=xtick_lbl, rotation='vertical')
    ax1.set_xlabel(xlabel)
    ax1.legend(loc='upper left', framealpha=0, bbox_to_anchor=bbox_loc_rat[var], fontsize=8)
    ax1.grid(alpha=0.3)
    
    ax2.set_ylim(ylimrat[var])
    ax2.set_yticks([])
    ax2.set_xticks(ticks=[0.5],labels=['ANNUAL'],rotation='vertical')
    ax2.grid(alpha=0.3)
    for tick in yticksrat[var]:
        ax2.axhline(tick,color='tab:gray',zorder=0,lw=0.5,alpha=0.3)

    SaveFig(fig, graph_type_str+'.ratio', 'monyr', var, note)

    plt.close(fig)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 6.44 µs


In [32]:
%%time

if plot_types['mtrd'] and subset_time:
    ## Make standard monthly trend plots + ratio of exp/obs trend explained plots in one figure
    # Set up 
    fig = plt.figure(figsize=(8,3.5), layout='constrained')
    gs = fig.add_gridspec(1, 5,  width_ratios=(6, 1,0.5,6, 1), wspace=0)
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    ax3 = fig.add_subplot(gs[3])
    ax4 = fig.add_subplot(gs[4])

    let_list = ['(a)', '(b)']

    # Plotting
    ax1.axhline(0,color='k',lw=0.5,zorder=0)
    ax2.axhline(0,color='k',lw=0.5,zorder=0)
    ax3.axhline(0,color='k',lw=0.5,zorder=0)
    ax4.axhline(0,color='k',lw=0.5,zorder=0)

    da_mon_era = ds_mon_trd['ERA5']
    da_ann_era = ds_ann_trd['ERA5']
    
    for dsname, attrs in ds_names.items():
        if attrs[0]:
            da_mon = ds_mon_trd[dsname]
            da_ann = ds_ann_trd[dsname]

            # LENS ensemble
            if attrs[1] == 0:
                # Extract mean & min and max of envelople
                da_mon_min = ds_mon_trd[dsname+' min']
                da_mon_max = ds_mon_trd[dsname+' max']

                da_ann_min = ds_ann_trd[dsname+' min']
                da_ann_max = ds_ann_trd[dsname+' max']
                
                # Plot envelope
                ax1.fill_between(da_mon_min[period].values, da_mon_min.values,
                    da_mon_max.values, color=attrs[2], alpha=0.1,
                    ec=None, zorder=attrs[4],label=ds_paper_names[dsname])
                ax2.axhspan(ymin=da_ann_min.values, ymax=da_ann_max.values,
                   xmin=0.45, xmax=0.55, alpha=0.1, color=attrs[2],ec=None,zorder=attrs[4])

            # PiC_UVnudge ensemble
            elif attrs[1] == 1:
                index = dict(ensemble_member=1)

                # Plot first ensemble member
                da_mon.loc[index].plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname])
                ((da_mon.loc[index]/da_mon_era)*100).plot(
                    ax=ax3, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+'/'+ds_paper_names['ERA5'])
                
                ax2.axhline(da_ann.loc[index].values, xmin=0.45, xmax=0.55, zorder=attrs[4], 
                            ls=attrs[3], color=attrs[2]) 
                ax4.axhline(((da_ann.loc[index]/da_ann_era)*100).values, xmin=0.45, xmax=0.55, zorder=attrs[4], 
                            ls=attrs[3], color=attrs[2]) 

                # Plot other ensemble members
                for i in range(2,4):
                    index['ensemble_member'] = i
            
                    da_mon.loc[index].plot(
                        ax=ax1, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                    ((da_mon.loc[index]/da_mon_era)*100).plot(
                        ax=ax3, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                    
                    ax2.axhline(da_ann.loc[index].values, xmin=0.45, xmax=0.55,
                           zorder=attrs[4], color=attrs[2], ls=attrs[3])
                    ax4.axhline(((da_ann.loc[index]/da_ann_era)*100).values, xmin=0.45, xmax=0.55,
                           zorder=attrs[4], color=attrs[2], ls=attrs[3])
         
            # PiC_UVnudge single run & Observations
            elif attrs[1] >= 2:
                # Plot monthly & annual trends
                da_mon.plot(
                    ax=ax1, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname])
                ax2.axhline(da_ann.values, xmin=0.45, xmax=0.55,
                    color=attrs[2], ls=attrs[3], zorder=attrs[4])

                if attrs[1] == 2:
                    ((da_mon/da_mon_era)*100).plot(
                        ax=ax3, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+'/'+ds_paper_names['ERA5'])
                    ax4.axhline(((da_ann/da_ann_era)*100).values, xmin=0.45, xmax=0.55,
                        color=attrs[2], ls=attrs[3], zorder=attrs[4])
                
    
    # Formatting
    ax1.set_title('')
    ax1.set_xlim(xlim)
    ax1.set_ylim(ylim[var])
    ax1.set_ylabel(ylabel[var])
    ax1.set_xticks(ticks=xtick_loc, labels=xtick_lbl, rotation='vertical')
    ax1.set_xlabel(xlabel)
    ax1.legend(loc='upper left', framealpha=0, bbox_to_anchor=bbox_loc[var], fontsize=8)
    ax1.grid(alpha=0.3)
    ax1.annotate(let_list[0],(0.15,0.93),xytext=(-10,0),ha='right',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')
    
    ax2.set_ylim(ylim[var])
    ax2.set_yticks([])
    ax2.set_xticks(ticks=[0.5],labels=['ANNUAL'],rotation='vertical')
    ax2.grid(alpha=0.3)
    for tick in yticks[var]:
        ax2.axhline(tick,color='tab:gray',zorder=0,lw=0.5,alpha=0.3)

    ax3.set_title('')
    ax3.set_xlim(xlim)
    ax3.set_ylim(ylimrat[var])
    ax3.set_ylabel(ylabelrat[var])
    ax3.set_xticks(ticks=xtick_loc, labels=xtick_lbl, rotation='vertical')
    ax3.set_xlabel(xlabel)
    ax3.legend(loc='upper left',framealpha=0, bbox_to_anchor=bbox_loc_rat[var], fontsize=8)
    ax3.grid(alpha=0.3)
    ax3.annotate(let_list[1],(0.15,0.93),xytext=(-10,0),ha='right',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')
    
    ax4.set_ylim(ylimrat[var])
    ax4.set_yticks([])
    ax4.set_xticks(ticks=[0.5],labels=['ANNUAL'],rotation='vertical')
    ax4.grid(alpha=0.3)
    for tick in yticksrat[var]:
        ax4.axhline(tick,color='tab:gray',zorder=0,lw=0.5,alpha=0.3)

    SaveFig(fig, graph_type_str, 'monyr-ratio', var, note)

    plt.close(fig)

CPU times: user 6 µs, sys: 0 ns, total: 6 µs
Wall time: 7.87 µs


#### Running trends plots
##### Attribution Contour plots

In [33]:
%%time

if plot_types['mtrd'] and not subset_time and ens_type and comp != 'ocn':
    # Framework
    frame = {'model': ['HIST+winds','Greenhouse gases','Non-GHG','Winds']}
    if var != 'MOC'  and var != 'AEROD_v':
        frame['model+obs'] = ['HIST+winds']
        for obs_name in obs_name_list:
            frame['model+obs'].append(obs_name)
            frame['model+obs'].append('Bias ('+obs_name[5:-1]+')')
    if r_domain:
        regions = ds_attr_mon.region.values
    else:
        regions = np.array(['A'])
        
    # Setup
    colPlot = 2

    for r in regions:
        rdict = dict(region=r) if r_domain else dict()
        dr_attr_mon = dsM_attr_mon.loc[rdict]
        dr_pval_mon = dsM_pval_mon.loc[rdict]
        addon = ''
    
        for f, fvlist in frame.items():
            df_attr_mon = dr_attr_mon[fvlist]
            df_pval_mon = dr_pval_mon[fvlist]
            
            numVar = len(df_attr_mon.data_vars)
            chgt = int(np.ceil(numVar/colPlot))
            totalPlot = colPlot*chgt
            hr = np.ones(chgt+1)
            hr[-1] = 0.1
            let_labels = FigLabelList(totalPlot)
            
            fig, axlist = plt.subplots(chgt+1, colPlot, height_ratios=hr, layout='constrained')
            fig.set_size_inches(ywidth*colPlot,3.5*chgt)
            axlist = axlist if chgt == 1 else list(itertools.chain.from_iterable(axlist))
            gs = axlist[-2].get_gridspec()
            for ax in axlist[-2:]:
                ax.remove()
            cbar_ax = fig.add_subplot(gs[-2:])
        
            for daname, da_attr in df_attr_mon.items():
                p = ds_attr_names[daname][0]
                da_pval = df_pval_mon[daname]

                patt_add = ','+str(round(float(Mpatt_corr_mon[daname]),2)) if 'OBS' in daname else ''
            
                # Plot data
                if 'NSIDC' not in daname and 'OSISAF' not in daname:
                    cax = da_attr.plot.contourf(
                            ax=axlist[p], x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                            add_colorbar=False, zorder=1)
                    # clab = da_attr.plot.contour(
                    #         ax=axlist[p], x=xdim, y=period, levels=mlabels[var],
                    #         colors='k', zorder=2)
                    da_pval.plot.contourf(
                        ax=axlist[p], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                        add_colorbar=False, zorder=2)
                    axlist[p].set_xlabel(xlabel, fontsize=14)
                    axlist[p].set_ylabel(date_str, fontsize=14)
                    axlist[p].set_yticks(ticks=np.arange(1,13), labels=m_str)
                    axlist[p].set_xlim(xlim)
                    axlist[p].set_title(daname+patt_add, fontsize=15)
                    axlist[p].tick_params(labelsize=12)
                    # axlist[p].clabel(clab, clab.levels, fontsize=12)
                    axlist[p].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')

                else:
                    axin = inset_axes(axlist[p], width='46.4%', height='100%', loc='center right', borderpad=0)
                    cax = da_attr.plot.contourf(
                            ax=axin, x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                            add_colorbar=False, zorder=1)
                    # clab = da_attr.plot.contour(
                    #         ax=axin, x=xdim, y=period, levels=mlabels[var],
                    #         colors='k', zorder=2)
                    da_pval.plot.contourf(
                            ax=axin, x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                            add_colorbar=False, zorder=2)
                    axin.set_xlabel(xlabel, fontsize=14)
                    axin.set_ylabel(date_str, fontsize=14)
                    axin.set_yticks(ticks=np.arange(1,13), labels=m_str)
                    axin.tick_params(labelsize=12)
                    # axin.clabel(clab, clab.levels, fontsize=12)
                    axin.set_xlim([1990,t_end-(trd_length-1)+10])
                    axin.set_title(daname+patt_add, fontsize=15)
                    axin.annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')
                    axlist[p].set_ylabel(' ', fontsize=14)
                    axlist[p].set_xlabel(' ', fontsize=14)
                    axlist[p].set_title(' ', fontsize=18)
                    axlist[p].spines['top'].set_visible(False)
                    axlist[p].spines['right'].set_visible(False)
                    axlist[p].spines['bottom'].set_visible(False)
                    axlist[p].spines['left'].set_visible(False)
                    axlist[p].tick_params(labelsize=12, length=0, width=0)
                    axlist[p].set_xticks(np.arange(0,1.1,1), labels=[' ',' '])
                    axlist[p].set_yticks(np.arange(0,1.1,1), labels=[' ',' '])
                    
                
        
            if numVar < totalPlot:
                axlist[1].axis('off')
        
            # Formatting
            cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='bottom')
            cb.set_label(label=ylabel[var], fontsize=14)
            cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=mticks[var])

            if r_domain:
                fig.suptitle(region_names[r],fontsize=16)
        
            SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-month-attribute.'+f+addon, var, note)
        
            plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.44 µs


##### AMOC Effect

In [34]:
%%time

# Load other variable
if plot_types['mtrd'] and not subset_time and ens_type and remove_amoc:
    var3 = 'aice' if var == 'TREFHT' else 'TREFHT'
    ds3_mon_trd = LoadData(graph_type_str+'.'+trdyr_str, var3, 'month')
    ds3_mon_trd = ds3_mon_trd.assign_coords(mid_year=('start_year',ds3_mon_trd.start_year.values+10))
    ds3_attr_mon = AttrCalc(ds3_mon_trd, var3, True)
    ds3_pval_mon = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_attr_mon)
    
    ds3_mon_trd_amoc, ds3_mon_trd_amoc_eff = DetrendAMOC(ds3_mon_trd, ds2_ann_trd, 0, var3)
    ds3_attr_mon_amoc = AttrCalc(ds3_mon_trd_amoc, var3, True)
    ds3_pval_mon_amoc = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_attr_mon_amoc)
    ds3_pval_mon_amoc_eff = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_mon_trd_amoc_eff)

CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 5.72 µs


In [35]:
%%time

if plot_types['mtrd'] and not subset_time and ens_type and remove_amoc:
    # Framework    
    frame = {'HIST+winds (with AMOC)': [0, 'HIST+winds',[ds_attr_mon,ds_pval_mon,
                                                         ds3_attr_mon,ds3_pval_mon]],
             'HIST+winds (without AMOC)': [1, 'HIST+winds',[ds_attr_mon_amoc,ds_pval_mon_amoc,
                                                            ds3_attr_mon_amoc,ds3_pval_mon_amoc]],
             'AMOC Effect': [2, 'HIST_UVnudge_LM',[ds_mon_trd_amoc_eff,ds_pval_mon_amoc_eff,
                                                   ds3_mon_trd_amoc_eff,ds3_pval_mon_amoc_eff]]}
    # Setup
    colPlot = 3
    numVar = 6
    chgt = int(np.ceil(numVar/colPlot))
    totalPlot = colPlot*chgt
    wr = np.ones(colPlot+1)
    wr[-1] = 0.05
    let_labels = FigLabelList(totalPlot)
    
    fig, axlist = plt.subplots(chgt, colPlot+1, width_ratios=wr, layout='constrained')
    fig.set_size_inches((ywidth*colPlot),3.5*chgt)
    gs = axlist[0,-1].get_gridspec()
    for ax in axlist[:,-1]:
        ax.remove()
    cbar_ax = fig.add_subplot(gs[0,-1])
    cbar_ax3 = fig.add_subplot(gs[1,-1])

    for label, flist in frame.items():
        c = flist[0]
        dname = flist[1]
        da = flist[2][0][dname]
        da_pval = flist[2][1][dname]
        da3 = flist[2][2][dname]
        da3_pval = flist[2][3][dname]
    
        # Plot data
        cax = da.plot.contourf(
                ax=axlist[0,c], x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                add_colorbar=False, zorder=1)
        # clab = da.plot.contour(
        #         ax=axlist[0,c], x=xdim, y=period, levels=mlabels[var],
        #         colors='k', zorder=2)
        # axlist[0,c].clabel(clab, clab.levels, fontsize=12)
        da_pval.plot.contourf(
                ax=axlist[0,c], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)

        cax3 = da3.plot.contourf(
                ax=axlist[1,c], x=xdim, y=period, levels=mlevels[var3], cmap=mcmaps[var3],
                add_colorbar=False, zorder=1)
        # clab3 = da3.plot.contour(
        #         ax=axlist[1,c], x=xdim, y=period, levels=mlabels[var3],
        #         colors='k', zorder=2)
        # axlist[1,c].clabel(clab3, clab3.levels, fontsize=12)
        da3_pval.plot.contourf(
                ax=axlist[1,c], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)
        
        for r in range(0,2):
            axlist[r,c].set_xlabel(xlabel, fontsize=14)
            axlist[r,c].set_ylabel(date_str, fontsize=14)
            axlist[r,c].set_yticks(ticks=np.arange(1,13), labels=m_str)
            axlist[r,c].set_xlim(xlim)
            axlist[r,c].set_title(label, fontsize=15)
            axlist[r,c].tick_params(labelsize=12)
            p = r*colPlot+c
            axlist[r,c].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')

    if numVar < totalPlot:
        for p in range(numVar,totalPlot):
            axlist[p].axis('off')

    # Formatting
    cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='right')
    cb.set_label(label=ylabel2line[var], fontsize=14)
    cb.ax.tick_params(labelsize=12)
    cb.set_ticks(ticks=mticks[var])

    cb3 = fig.colorbar(cax3, cax=cbar_ax3, shrink=0.1, extend='both', location='right')
    cb3.set_label(label=ylabel2line[var3], fontsize=14)
    cb3.ax.tick_params(labelsize=12)
    cb3.set_ticks(ticks=mticks[var3])

    SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-month-amoc', var+'-'+var3, note)

    plt.close(fig)

CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 13.8 µs


##### AMOC trends by latitude

In [36]:
%%time

if plot_types['mtrd'] and not subset_time and ens_type and comp == 'ocn' and o_domain:
    # Framework
    frame = {'model': ['HIST+winds','Greenhouse gases','Non-GHG','Winds']}
        
    # Setup
    colPlot = 2
    
    for f, fvlist in frame.items():
        df_attr_ann = ds_attr_ann[fvlist]
        df_pval_ann = ds_pval_ann[fvlist]
        
        numVar = len(df_attr_ann.data_vars)
        chgt = int(np.ceil(numVar/colPlot))
        totalPlot = colPlot*chgt
        hr = np.ones(chgt+1)
        hr[-1] = 0.1
        let_labels = FigLabelList(totalPlot)
        
        fig, axlist = plt.subplots(chgt+1, colPlot, height_ratios=hr, layout='constrained')
        fig.set_size_inches(ywidth*colPlot,3.5*chgt)
        axlist = axlist if chgt == 1 else list(itertools.chain.from_iterable(axlist))
        gs = axlist[-2].get_gridspec()
        for ax in axlist[-2:]:
            ax.remove()
        cbar_ax = fig.add_subplot(gs[-2:])
    
        for daname, da_attr in df_attr_ann.items():
            p = ds_attr_names[daname][0]
            da_pval = df_pval_ann[daname]
        
            # Plot data
            cax = da_attr.plot.contourf(
                    ax=axlist[p], x=xdim, y='lat', levels=mlevels[var], cmap=mcmaps[var],
                    add_colorbar=False, zorder=1)
            # clab = da_attr.plot.contour(
            #         ax=axlist[p], x=xdim, y=period, levels=mlabels[var],
            #         colors='k', zorder=2)
            da_pval.plot.contourf(
                ax=axlist[p], x=xdim, y='lat', levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)
            axlist[p].set_xlabel(xlabel, fontsize=14)
            axlist[p].set_ylabel('Latitude', fontsize=14)
            axlist[p].set_yticks(ticks=np.arange(20,61,10))
            axlist[p].set_xlim(xlim)
            axlist[p].set_title(daname, fontsize=15)
            axlist[p].tick_params(labelsize=12)
            # axlist[p].clabel(clab, clab.levels, fontsize=12)
            axlist[p].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')           
            
    
        if numVar < totalPlot:
            axlist[1].axis('off')
    
        # Formatting
        cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='bottom')
        cb.set_label(label=ylabelsingle [var], fontsize=14)
        cb.ax.tick_params(labelsize=12)
        cb.set_ticks(ticks=mticks[var])
    
        SaveFig(fig, graph_type_str, trdyr_str+'-trend-month-attribute.'+f, var, note)
    
        plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


##### Second-order plots

In [37]:
## Second order plots - TBD format 

# %%time

# if plot_types['mtrd'] and not subset_time and ens_type and var != 'MOC':
#     # Framework
#     frame = {'2nd-order': ['wf_ft','fw_wt','gw_wt','nw_wt']}
        
#     # Setup
#     colPlot = 3
    
#     for f, fvlist in frame.items():
#         df_mon_2nd = ds_mon_2nd[fvlist]
#         numVar = len(df_mon_2nd.data_vars)
#         chgt = int(np.ceil(numVar/colPlot))
#         totalPlot = colPlot*chgt
        
#         fig, axlist = plt.subplots(chgt, colPlot, layout='constrained')
#         fig.set_size_inches(ywidth[t_domain],3.5*chgt)
#         axlist = axlist if chgt == 1 else list(itertools.chain.from_iterable(axlist))
    
#         for daname, da_attr in df_mon_2nd.items():
#             p = ds_attr_names[daname][0]
        
#             # Plot data
#             cax = da_attr.plot.contourf(
#                     ax=axlist[p], x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
#                     add_colorbar=False, zorder=1)
#             axlist[p].set_xlabel(xlabel)
#             axlist[p].set_ylabel(date_str)
#             axlist[p].set_yticks(ticks=np.arange(1,13), labels=m_str)
#             axlist[p].set_xlim(xlim)
#             axlist[p].set_title(daname)
    
#         if numVar < totalPlot:
#             for p in range(numVar,totalPlot):
#                 axlist[p].axis('off')
    
#         # Formatting
#         cb = fig.colorbar(cax, ax=axlist[:],shrink=0.75, pad=0.05,fraction=0.05, extend='both')
#         cb.set_label(label=ylabel[var], fontsize=12)
#         cb.ax.tick_params(labelsize=12)
#         cb.set_ticks(ticks=mticks[var])
    
#         SaveFig(fig, graph_type_str, '20yr-trend-month-attribute.'+f, var, note)
    
#         plt.close(fig)

##### Linear plot of 20-year trends

In [38]:
# %%time

# if plot_types['mtrd'] and not subset_time:
#     # Setup
#     fig, ax = plt.subplots(1, 1, layout='constrained')
#     fig.set_size_inches(6,4)

#     if var == 'TREFHT' or var == 'MOC' or var == 'AEROD_v':
#         ds_trd = dsM_ann_trd
#         addon = '.year'
#     elif var == 'aice':
#         ds_trd = dsM_mon_trd.loc[dict(month=type_onemonth)]
#         addon = '.'+mon_str[type_onemonth-1]
    
#     # Plot data
#     # Background lines
#     ax.axhline(0, color='k', zorder=0, lw=0.5)
    
#     for dsname, attrs in ds_names.items():
#         if attrs[0]:
#             da_trd = ds_trd[dsname]
#             # PIcontrol ensemble
#             if attrs[1] == 0:
#                 le_std = ds_trd[dsname].std('slice')*2
#                 lb = ds_trd[dsname+' mean']-le_std
#                 ub = ds_trd[dsname+' mean']+le_std
                
#                 ax.fill_between(lb[xdim].values, lb.values, ub.values,
#                                 color=attrs[2], alpha=0.1, ec=None, zorder=attrs[4],label=ds_paper_names[dsname])    

#             # Nudging ensemble
#             elif attrs[1] == 1 and not ens_type:
#                 index = dict(ensemble_member=1)

#                 # Plot first ensemble member
#                 da_trd.loc[index].plot(
#                     ax=ax, color=attrs[2], ls=attrs[3], x=xdim, zorder=attrs[4], label=ds_paper_names[dsname]) 

#                 # Plot other ensemble members
#                 for i in range(2,4):
#                     index['ensemble_member'] = i
            
#                     da_trd.loc[index].plot(
#                         ax=ax, color=attrs[2], x=xdim, zorder=attrs[4], ls=attrs[3])              

#             # Nudging ensemble mean or observation
#             elif (attrs[1] == 1 and ens_type) or attrs[1] == 3:
#                 da_trd.plot(
#                     ax=ax, color=attrs[2], ls=attrs[3], x=xdim, zorder=attrs[4], label=ds_paper_names[dsname])                 

#     # Formatting
#     ax.grid(alpha=0.3)
#     ax.set_xlim(xlim)
#     ax.set_ylim(ylim[var])
#     ax.set_ylabel(ylabelsingle[var])
#     ax.set_xlabel(xlabel)
#     ax.set_title('')
#     ax.legend(loc=bbox_loc[var],framealpha=0, fontsize=8)

#     SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend'+addon, var, note)

#     plt.close(fig)

##### Linear attribution plots

In [39]:
# %%time

# if plot_types['mtrd'] and not subset_time:
#     if var == 'TREFHT' or var == 'MOC' or var == 'AEROD_v':
#         ds_attr_trd = dsM_attr_ann
#         addon = '.year'
#     elif var == 'aice':
#         ds_attr_trd = dsM_attr_mon.loc[dict(month=type_onemonth)]
#         addon = '.'+mon_str[type_onemonth-1]

#     attr_order_sets = {'w-g-n-S': ['Winds', 'Greenhouse Gases', 'Non-GHG', 'HIST+Winds']}
#     if var != 'MOC' and var != 'AEROD_v':
#         attr_order_sets['S-R-o'] = ['HIST+Winds', 'Bias', ds_paper_names[obs_name]]

#     # Winds, IV no wind, GHG, nonGHG (or hist), Obs
#     ds_attr_neg = xr.where(ds_attr_trd < 0, ds_attr_trd, 0.0)
#     ds_attr_pos = xr.where(ds_attr_trd > 0, ds_attr_trd, 0.0)

#     for anote, aorder in attr_order_sets.items():
#         # Setup
#         fig, ax = plt.subplots(1, 1, layout='constrained')
#         fig.set_size_inches(6,4)

#         # Plot data
#         # Background lines
#         ax.axhline(0, color='k', zorder=100, lw=0.5)
    
#         for i in range(0,len(aorder)-1):
#             daname = aorder[i]
#             da_attr_neg = ds_attr_neg[daname]
#             da_attr_pos = ds_attr_pos[daname]
#             dcolor = ds_attr_names[daname][1]
#             if i == 0:
#                 ax.fill_between(da_attr_neg.start_year.values,da_attr_neg.values,color=dcolor,
#                                 ec='None',label=daname)
#                 ax.fill_between(da_attr_pos.start_year.values,da_attr_pos.values,color=dcolor,
#                                 ec='None')
#             else:
#                 da_attr_neg = da_attr_neg+da_attr_neg_prev
#                 da_attr_pos = da_attr_pos+da_attr_pos_prev
#                 ax.fill_between(da_attr_neg.start_year.values,da_attr_neg_prev.values,
#                                 da_attr_neg.values,color=dcolor,ec='None',label=daname)
#                 ax.fill_between(da_attr_pos.start_year.values,da_attr_pos_prev.values,
#                                 da_attr_pos.values,color=dcolor,ec='None')
                
#             da_attr_neg_prev = da_attr_neg
#             da_attr_pos_prev = da_attr_pos

#         ds_attr_trd[aorder[-1]].plot(ax=ax,color='k', zorder=100, lw=1,label=aorder[-1])
        
    
#         # Formatting
#         ax.grid(alpha=0.3)
#         ax.set_xlim(xlim)
#         ax.set_ylim(ylimcontr[var])
#         ax.set_ylabel(ylabelsingle[var])
#         ax.set_xlabel(xlabel)
#         ax.set_title('')
#         ax.legend(framealpha=0, fontsize=8)
    
#         SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-contribution-'+anote+addon, var, note)
    
#         plt.close(fig)
#         ax.clear()

##### 2-variable scatter plots

In [40]:
# %%time

# # Secondary variable
# if plot_types['mtrd'] and not subset_time:
#     ann_varlist = ['TREFHT','MOC']
#     mon_varlist = ['aice','AEROD_v']

#     ds_trd = dsM_ann_trd if var in ann_varlist else dsM_mon_trd.loc[dict(month=type_onemonth)]
#     ds2_trd = ds2_ann_trd if var2 in ann_varlist else ds2_mon_trd.loc[dict(month=type_onemonth)]
    
#     free_le_names = ['LENS2 piControl','GHG2','LENS2 HIST-SSP370']
#     free_le_count = {}
#     ms_x,ms_y = np.meshgrid(np.linspace(ylim[var2][0],ylim[var2][1],50),np.linspace(ylim[var][0],ylim[var][1],50))
    
#     for name in free_le_names:
#         ms_count = np.zeros_like(ms_x)

#         for y in ds_trd.start_year.values:
#             da_trd = ds_trd.loc[{'start_year': y}]
#             da2_trd = ds2_trd.loc[{'start_year': y}]

#             x_select = np.logical_and(ms_x >= da2_trd[name+' min'].values, ms_x <= da2_trd[name+' max'].values)
#             y_select = np.logical_and(ms_y >= da_trd[name+' min'].values, ms_y <= da_trd[name+' max'].values)
#             tot_select = np.logical_and(x_select, y_select)
            
#             ms_count[tot_select] += 1
#         free_le_count[name] = ms_count

In [41]:
# %%time

# # Variable 1 (y-axis) vs Variable 2 (x-axis) for freely evolving & nudged runs
# if plot_types['mtrd'] and not subset_time and not r_domain:
#     if var in mon_varlist or var2 in mon_varlist:
#         addon = '.'+mon_str[type_onemonth-1]
#     else:
#         addon = '.year'
        
#     # Framework
#     comp_list = ['LENS2 piControl', 'PiC_UVnudge_LM2006', 'd',
#                 'GHG2', 'GHG_UVnudge_LM', 'd',
#                 'LENS2 HIST-SSP370', 'HIST_UVnudge_LM','d']

#     # Setup
#     hgt = 3
#     wdth = 3
#     fig, axlist = plt.subplots(hgt, wdth, layout='constrained')
#     fig.set_size_inches(4*wdth,3*hgt)
#     axlist = axlist.flatten()
#     axct = 0

#     for fname in comp_list:
#         pos = axct % 3
#         if fname == 'd':
#             da_trd = ds_trd[comp_list[axct-1]]-ds_trd[comp_list[axct-2]+' mean']
#             da2_trd = ds2_trd[comp_list[axct-1]]-ds2_trd[comp_list[axct-2]+' mean']
#         else:
#             fname_i = fname+' mean' if pos == 0 else fname
#             da_trd = ds_trd[fname_i]
#             da2_trd = ds2_trd[fname_i]
        

#         r_val = xr.corr(da2_trd, da_trd, dim='start_year')
#         r_lab = '$\it{r}$'+'={:.2f}; '.format(r_val.values)
#         r_lab = r_lab+'$\it{r}^2$'+'={:.2f}'.format(r_val.values**2)
    
#         # Plot data
#         cax = axlist[axct].scatter(da2_trd.values, da_trd.values, c=da_trd.start_year.values, 
#                                    cmap=cm.lajolla_r, edgecolors='k',label=r_lab,zorder=2)
#         if fname != 'd':
#             lename = fname if pos == 0 else comp_list[axct-1]
#             ucax = axlist[axct].contourf(ms_x,ms_y,free_le_count[lename],vmin=0,vmax=75,cmap=cm.nuuk,zorder=0)

#         axlist[axct].set_xlabel(ylabeltwolinesingle[var2])
#         axlist[axct].set_ylabel(ylabeltwolinesingle[var])
#         axlist[axct].set_ylim(ylim[var])
#         axlist[axct].set_xlim(ylim[var2])
#         if pos < 2:
#             axlist[axct].set_title(ds_paper_names[fname])
#         else:
#             axlist[axct].set_title(ds_paper_names[comp_list[axct-1]]+' – '+ds_paper_names[comp_list[axct-2]])
#         axlist[axct].legend(framealpha=0, fontsize=8)
#         axlist[axct].grid(alpha=0.3)
#         axlist[axct].axhline(y=0,xmin=-20,xmax=20,color='k',lw=0.7,zorder=1)
#         axlist[axct].axvline(x=0,ymin=-20,ymax=20,color='k',lw=0.7,zorder=1)

#         axct +=1 

#     # Formatting
#     cb = fig.colorbar(cax, ax=axlist[:],shrink=0.5, pad=0.05,fraction=0.02, extend='both',location='bottom')
#     cb.set_label(label=xlabel, fontsize=10)
#     cb.ax.tick_params(labelsize=10)

#     # cb2 = fig.colorbar(ucax, ax=axlist[:],shrink=0.5, pad=0.05,fraction=0.02, extend='both',location='bottom')
#     # cb2.set_label(label='Freely-evolving ensemble min-max bounds', fontsize=10)
#     # cb2.ax.tick_params(labelsize=10)

#     SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-Free-v-Nudge'+addon, var+'-'+var2, note)

#     plt.close(fig)

## Line plots
### Set up

In [42]:
if plot_types['line']:
    graph_type_str = 'Linear'

    ylabelabs = {'aice': 'Sea ice extent (million km$^2$)',
                 'TREFHT': '2m air temperature (K)',
                 'RESTOM': 'Net TOA Imbalance (W/m$^{2}$)',
                 'MOC': 'Atlantic Meridional Overturning Circulation (Sv)',
                 'TGCLDLWP': 'Liquid water path (g/m$^{2}$)',
                 'AEROD_v': 'Aerosol optical depth',
                 'FSNTOA': 'Net TOA shortwave flux (W/m$^{2}$)'}
    ylabeldtrd = {'TREFHT': 'Detrended 2m air temperature anomaly (K)'}
    ylabelensp = {'TREFHT': '2m air temperature ensemble spread (K)'}
    ylabel1m = {'aice': ' sea ice extent (million km$^2$)',
                'TREFHT': ' 2m air temperature (K)',
                'TGCLDLWP': ' liquid water path (g/m$^{2}$)',
                'AEROD_v': ' aerosol optical depth'}
    ylabelanom = {'TREFHT': '2m air temperature anomaly (K)',
                  'aice': month_str[type_onemonth-1]+' sea ice extent anomaly (million km$^2$)',
                 'MOC': 'Atlantic Meridional Overturning\nCirculation anomaly (Sv)'}
    ylimabs = {'aice': {3: [13,17],
                        9: [3,10]},
               'TREFHT': [[286.5,289.1],[254,265]],
               'AEROD_v': [0,0.05]}
    ylimanom = {'aice': {3: [-1.5,3],
                        9: [-5,3]},
               'TREFHT': [[-2,3],[-5,7]]}

    wdth = {1950: 6,
           1980: 6}

    legend_loc = 'best'

    double_plot = (type_onemonth == 39 and var == 'aice')
    ramoc_str = '.DtrdAMOC' if remove_amoc else ''

    # Prepare ylim
    if var in ylimabs:
        if var == 'aice' and type_onemonth < 13:
            ylim = ylimabs[var][type_onemonth]
            yliman = ylimanom[var][type_onemonth]
        elif var == 'TREFHT':
            ylim = ylimabs[var][s_domain]
            yliman = ylimanom[var][s_domain]
        elif var == 'AEROD_v':
            ylim = ylimabs[var]
        else:
            ylim = False

    xtick_loc = np.arange(t_domain,2025,5)
    xtick_loc_min = np.arange(t_domain,2025,1)
    xtick_lbl = np.arange(t_domain,2025,5)
    xlim = [t_domain,2024]
    xlabel= 'Year'
    
    # Prepare x ticks
    # Timeseries
    if time_avg == 4:
        period ='time'
        attr_avg_time = {period: slice('1950-01-01','1970-12-31')}        

        if type_onemonth < 13:
            date_str = mon_str[type_onemonth-1]
            ylabelabs[var] = month_str[type_onemonth-1]+ylabel1m[var]
        else:
            date_str = mon_str[3-1]+'-'+mon_str[9-1]
            ylabelabs[var] = {3: month_str[3-1]+ylabel1m[var],
                             9:month_str[9-1]+ylabel1m[var]}
        
        
    # Yearly
    elif time_avg == 1:
        date_str = 'timeseries_yr'
        period='year'
        dim_avg = 'time.year'
        attr_avg_time = {period: slice(1950,1970)}

        # Regional averages
        if r_domain:
            xtick_loc = np.arange(t_domain,2025,10)
            xtick_loc_min = np.arange(t_domain,2025,5)
            xtick_lbl = np.arange(t_domain,2025,10)

    # Seasonal
    elif time_avg == 2:
        period = 'season'
        attr_avg_time = {'year': slice(1950,1970)}
        date_str = mon_seas_dict[type_onemonth]

### Data loading

In [43]:
%%time

if plot_types['line']:

    ## Absolute (only one that will be used for SIA-one month, TOA, AMOC, drift)
    # Yearly
    if time_avg == 1:
        ds_abs = LoadData(graph_type_str+'.abs', var, period)
        pimean = ds_abs['PiC_UVnudge_LM2006'].loc[attr_avg_time].mean(period)
        ds_attr = AttrCalc(ds_abs-pimean, var, False)

        # # For temperature plots only
        # if var == 'TREFHT':
        #     # Absolute correlation coefficients
        #     ds_abs_r = LoadData(graph_type_str+'.absR', var, period)    

        #     # Detrended anomalies correlation coefficients
        #     ds_dtrd_r = LoadData(graph_type_str+'.anomdtrdR', var, period)    

    # Timeseries of one month of sea ice area data
    elif time_avg == 4:
        # If only plotting one month
        if type_onemonth < 13:
            ds_abs = LoadData(graph_type_str+'.abs', var, mon_str[type_onemonth-1])
            pimean = ds_abs['PiC_UVnudge_LM2006'].loc[attr_avg_time].mean(period)
            ds_attr = AttrCalc(ds_abs-pimean, var, False)

            # if use_era5:
            #     # Absolute correlation coefficients
            #     ds_abs_r = LoadData(graph_type_str+'.absR', var, mon_str[type_onemonth-1])

    elif time_avg == 2:
        ds_abs = LoadData(graph_type_str+'.abs', var, period)
        ds_abs = ds_abs.loc[dict(season=date_str)]
        pimean = ds_abs['PiC_UVnudge_LM2006'].loc[attr_avg_time].mean('year')
        ds_attr = AttrCalc(ds_abs-pimean, var, False)
        period='year'
    
    # LWP
    if var == 'TGCLDLWP':
        ds_abs = ds_abs*1000

Linear.abs.aice.Arctic.1950.Sep.All_members.nc
Non-GHG
  Calculated
Anthropogenic forcing
  Calculated
Greenhouse gases
  Calculated
Winds
  Calculated
HIST+winds
  Calculated
HIST-control
  Calculated
Greenhouse gases-control
  Calculated
Non-GHG-control
  Calculated
OBS (ERA5)
  Calculated
Bias (ERA5)
  Calculated
OBS (NSIDC)
  Calculated
Bias (NSIDC)
  Calculated
OBS (OSISAF)
  Calculated
Bias (OSISAF)
  Calculated
CPU times: user 934 ms, sys: 233 ms, total: 1.17 s
Wall time: 1.52 s


In [44]:
%%time

# Remove effect of AMOC
if plot_types['line'] and remove_amoc:
    ds2_abs = LoadData(graph_type_str+'.abs', 'MOC', 'year')
    
    # Remove AMOC 
    ds_abs_amoc = DetrendAMOC(ds_abs, ds2_abs, 1, var)
    pimean = ds_abs_amoc['PiC_UVnudge_LM2006'].loc[attr_avg_time].mean(period)
    ds_attr_amoc = AttrCalc(ds_abs_amoc-pimean, var, False)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.68 µs


In [45]:
# Set variables according to AMOC removal status
if plot_types['line']:
    if remove_amoc:
        dsM_abs = ds_abs_amoc
        dsM_attr = ds_attr_amoc
    else:
        dsM_abs = ds_abs
        dsM_attr = ds_attr

### Make absolute plots

In [46]:
%%time

if plot_types['line'] and not double_plot and not r_domain:
    # Setup figure
    fig,ax = plt.subplots(1,1, layout='constrained')
    fig.set_size_inches(wdth[t_domain],3.5)

    for dsname, attrs in ds_names.items():
        if attrs[0] and dsname != 'GISTEMP':
            da_abs = dsM_abs[dsname]

            # LENS ensemble
            if attrs[1] == 0:
                # Extract mean & min and max of envelope
                da_abs_mean = dsM_abs[dsname+' mean']
                da_abs_std = dsM_abs[dsname].std('slice')
                da_abs_min = da_abs_mean-da_abs_std
                da_abs_max = da_abs_mean+da_abs_std
                
                # Plot envelope
                ax.fill_between(da_abs_min[period].values, da_abs_min.values,
                    da_abs_max.values, color=attrs[2], alpha=0.1,
                    ec=None,label=ds_paper_names[dsname], zorder=attrs[4])
                
                # Plot mean
                da_abs_mean.plot(
                    ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4],label=ds_paper_names[dsname]+' mean')

            # PiC_UVnudge ensemble
            elif attrs[1] == 1 and not ens_type:
                index = dict(ensemble_member=1)
                addon = ''
                # if use_era5:
                #     addon = ' ($\it{r}^2$='+'{:.2f}; {:.2f})'.format(np.mean(ds_abs_r[dsname].values)**2, np.mean(ds_dtrd_r[dsname].values)**2) if var == 'TREFHT' else ''
                #     addon = ' ($\it{r}^2$='+'{:.2f})'.format(np.mean(ds_abs_r[dsname].values)**2) if var == 'aice' else addon
                # else:
                #     addon = ''
                    
                # Plot first ensemble member
                da_abs.loc[index].plot(
                    ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+addon)

                # Plot other ensemble members
                for i in range(2,4):
                    index['ensemble_member'] = i
            
                    da_abs.loc[index].plot(
                        ax=ax, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                    
            # PiC_UVnudge single run & Observations (& PiC_UVensemble means)
            elif attrs[1] >= 2 or (ens_type and attrs[1] == 1):
                addon = ''
                # if use_era5:
                #     addon = ' ($\it{r}^2$='+'{:.2f}; {:.2f})'.format(ds_abs_r[dsname].values**2, ds_dtrd_r[dsname].values**2) if ((var == 'TREFHT') and attrs[1] != 3) else ''
                #     addon = ' ($\it{r}^2$='+'{:.2f})'.format(ds_abs_r[dsname].values**2) if ((var == 'aice') and attrs[1] != 3) else addon
                # else:
                #     addon = ''
               
                # Plot absolute value
                da_abs.plot(
                    ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+addon)
            

    # Formatting
    ax.set_xlabel(xlabel)
    if time_avg == 1:
        ax.set_xlim(xlim)
        ax.set_xticks(ticks=xtick_loc,labels=xtick_lbl)
        ax.set_xticks(ticks=xtick_loc_min,minor=True)
    elif time_avg == 4:
        ax.xaxis.set_major_locator(mdates.YearLocator(5,month=type_onemonth))
        ax.xaxis.set_minor_locator(mdates.YearLocator(1,month=type_onemonth))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
    if var == 'AEROD_v':
        ax.set_yscale('log')
    elif ylim:
        ax.set_ylim(ylim)
    ax.set_ylabel(ylabelabs[var])
    ax.legend(fontsize=8, edgecolor='w', loc=legend_loc)
    ax.set_title('')
    ax.grid(alpha=0.3)

    SaveFig(fig, graph_type_str+ramoc_str+'.abs', date_str, var, note)

    plt.close(fig)

CPU times: user 633 ms, sys: 26.9 ms, total: 660 ms
Wall time: 762 ms


### Make absolute attribution plots

In [47]:
%%time

if plot_types['line'] and not double_plot and not r_domain:
    frame = {'model': ['HIST+Winds','Greenhouse Gases','Non-GHG','Winds'],
              'model+LE': ['HIST+Winds','Greenhouse Gases','Non-GHG','Winds','HIST-control','Greenhouse Gases-control','Non-GHG-control']} 
    # Setup figure
    for f, fvlist in frame.items():
        df_abs = dsM_attr[fvlist]
        
        fig,ax = plt.subplots(1,1, layout='constrained')
        fig.set_size_inches(wdth[t_domain],3.5)
        ax.axhline(0,color='k',lw=0.5,zorder=0)
        
        for daname, da_abs in df_abs.items():
            da_abs = df_abs[daname]
            attrs = ds_attr_names[daname]
            lename = '-control' in daname
            ls = '--' if lename else '-'
                
            if ens_type or lename:
                da_abs = da_abs.sel(ensemble_member=1) if lename else da_abs
                da_abs.plot(
                        ax=ax, color=attrs[1], x=period, label=daname,ls=ls)
            else:
                da_abs_mean = da_abs.mean('ensemble_member')
                da_abs_std = da_abs.std('ensemble_member')*2
                da_abs_min = da_abs_mean-da_abs_std
                da_abs_max = da_abs_mean+da_abs_std
                
                # Plot envelope
                ax.fill_between(da_abs_min[period].values, da_abs_min.values,
                    da_abs_max.values, color=attrs[1], alpha=0.2,
                    ec=None)
                
                # Plot mean
                da_abs_mean.plot(
                    ax=ax, color=attrs[1], x=period,label=daname,ls=ls)
        # Formatting
        ax.set_xlabel(xlabel)
        if time_avg == 1:
            ax.set_xlim(xlim)
            ax.set_xticks(ticks=xtick_loc,labels=xtick_lbl)
            ax.set_xticks(ticks=xtick_loc_min,minor=True)
        elif time_avg == 4:
            ax.xaxis.set_major_locator(mdates.YearLocator(5,month=type_onemonth))
            ax.xaxis.set_minor_locator(mdates.YearLocator(1,month=type_onemonth))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
        ax.set_ylabel(ylabelanom[var])
        ax.set_ylim(yliman)
        ax.legend(fontsize=8, edgecolor='w', loc=legend_loc)
        ax.set_title('')
        ax.grid(alpha=0.3)
    
        SaveFig(fig, graph_type_str+ramoc_str+'.anom-'+f, date_str, var, note)
    
        plt.close(fig)

KeyError: 'HIST+Winds'

### Make regional average plots

In [48]:
%%time

if plot_types['line'] and r_domain:
    # Setup figure
    fig,axlist = plt.subplots(3,2, layout='constrained')
    fig.set_size_inches(wdth[t_domain]*2,3.5*3)
    axlist = axlist.flatten()
    axct = 0

    # Cycle through all regions
    region_list = ds_abs.region.values
    for reg in region_list:
        reg_ind = dict(region=reg)
        dr_abs = ds_abs.loc[reg_ind]
        dr_abs_r = ds_abs_r.loc[reg_ind]
        dr_dtrd_r = ds_dtrd_r.loc[reg_ind]
        ax = axlist[axct]

        for dsname, attrs in ds_names.items():
            if attrs[0] and dsname != 'GISTEMP':
                da_abs = dr_abs[dsname]
    
                # LENS ensemble
                if attrs[1] == 0:
                    # Extract mean & min and max of envelope
                    da_abs_mean = dr_abs[dsname+' mean']
                    da_abs_min = dr_abs[dsname+' min']
                    da_abs_max = dr_abs[dsname+' max']
                    
                    # Plot envelope
                    ax.fill_between(da_abs_min[period].values, da_abs_min.values,
                        da_abs_max.values, color=attrs[2], alpha=0.1,
                        ec=None,label=ds_paper_names[dsname], zorder=attrs[4])
                    
                    # Plot mean
                    da_abs_mean.plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4],label=ds_paper_names[dsname]+' mean')
    
                # PiC_UVnudge ensemble
                elif attrs[1] == 1 and not ens_type:
                    index = dict(ensemble_member=1)
                    addon = ''
                        
                    # Plot first ensemble member
                    da_abs.loc[index].plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+addon)
    
                    # Plot other ensemble members
                    for i in range(2,4):
                        index['ensemble_member'] = i
                
                        da_abs.loc[index].plot(
                            ax=ax, color=attrs[2], x=period, zorder=attrs[4], ls=attrs[3])
                        
                # PiC_UVnudge single run & Observations (& PiC_UVensemble means)
                elif attrs[1] >= 2 or (ens_type and attrs[1] == 1):
                    addon = ''
                   
                    # Plot absolute value
                    da_abs.plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=period, zorder=attrs[4], label=ds_paper_names[dsname]+addon)
                
    
        # Formatting
        ax.set_xlabel(xlabel)
        if time_avg == 1:
            ax.set_xlim(xlim)
            ax.set_xticks(ticks=xtick_loc,labels=xtick_lbl)
            ax.set_xticks(ticks=xtick_loc_min,minor=True)
        elif time_avg == 4:
            ax.xaxis.set_major_locator(mdates.YearLocator(5,month=type_onemonth))
            ax.xaxis.set_minor_locator(mdates.YearLocator(1,month=type_onemonth))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
            ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
        if ylim:
            ax.set_ylim(ylim)
        ax.set_ylabel(ylabelabs[var])
        ax.set_title(region_names[reg])
        ax.grid(alpha=0.3)

        axct += 1

    axlist[4].legend(fontsize=11, edgecolor='w', bbox_to_anchor=(0, -0.8), loc='lower left')
    SaveFig(fig, graph_type_str+'.Reg', date_str, var, note)

    plt.close(fig)

CPU times: user 21 µs, sys: 0 ns, total: 21 µs
Wall time: 8.11 µs


In [49]:
varnum = 0
for dsname, attrs in ds_names.items():
    varnum = varnum+1 if attrs[0] else varnum

## Line and running trends plots
### Set up

In [50]:
if plots['comb'][0]:
    graph_type_str = 'Linear.Trend.abs-20yr-40yr'
    hgt = 3
    wdth = 3

    if var == 'MOC' and o_domain and not ens_type:
        graph_type_str += '.'+str(ocn_lat)+'N'
    elif var == 'MOC' and o_domain and ens_type:
        graph_type_str += '.20-55N'

    ylabelabs = {'TREFHT': 'Annual 2m air\ntemperature (K)',
                 'MOC': 'AMOC (Sv)',}
    ylabelabs1m = {'aice': ' sea\nice extent (million km$^2$)',}
    ylimabs = {'aice': {3: [13,17],
                        9: [0,10]},
               'TREFHT': [254,265],
               'MOC': [10,28]}
    yticksabs = {'aice': {3: np.arange(13,18,1),
                          9: np.arange(0,11,2)},
                'TREFHT': np.arange(254,266,3),
                'MOC': np.arange(10,28,5)}
    
    # Prepare x ticks
    # Timeseries
    if time_avg == 4:
        period ='time'
        date_str = mon_str[type_onemonth-1]
        
        xlimabs = [t_domain,2025]
        xlabelabs = 'Year' 
        ylabelabs[var] = month_str[type_onemonth-1]+ylabelabs1m[var]
        if var == 'aice':
            ylimabs['aice'] = ylimabs['aice'][type_onemonth]
            yticksabs['aice'] = yticksabs['aice'][type_onemonth]
        
    # Yearly
    elif time_avg == 1:
        period ='year'
        date_str = 'year'
        
        xlimabs = [t_domain,2024]
        xlabelabs= 'Year'

    xticks_loc = np.arange(1950,2021,10)
    xticks_locmin = np.arange(1950,2025,5)
    xlimtrd = lambda tl: [t_domain+10,t_end-(tl-1)+10]
    ylimtrd20 = {'aice': [-2,1],
           'TREFHT': [-2,2],
           'MOC': [-4,4],}
    ylimtrd40 = {'aice': [-1.2,0.5],
           'TREFHT': [-0.5,1.2],
           'MOC': [-3,3],}
    ywidth = lambda tl: ((t_end-(tl-1))-t_domain)*(3/55) # 20, 40, 1 (abs)
    ylabeltwolinesingle = {'aice': mon_str[type_onemonth-1]+' sea ice extent\ntrend (million km$^2$/decade)',
                    'TREFHT': 'Annual 2m air temperature\ntrend (K/decade)',
                   'MOC': 'AMOC (Sv/decade)',}
    xlabeltrd = lambda tl: 'Mid year of '+str(tl)+'-year trend'
    xdim = 'mid_year'

    # Plot letter
    plot_let_full = np.array(['(a)', '(b)', '(c)', '(d)',
                              '(e)', '(f)', '(g)', '(h)',
                              '(i)', '(j)', '(k)', '(l)',
                              '(m)', '(n)', '(o)', '(p)'])
    plot_let = plot_let_full[0:(wdth*hgt)].reshape((hgt, wdth))

### Data loading

In [51]:
%%time

if plots['comb'][0]:
    ## Absolute (only one that will be used for SIA-one month, TOA, AMOC, drift)
    # Yearly
    if time_avg == 1:
        ds_abs = LoadData('Linear.abs', var, period)

    # Timeseries of one month of sea ice area data
    elif time_avg == 4:
        # If only plotting one month
        ds_abs = LoadData('Linear.abs', var, mon_str[type_onemonth-1])

    ## Load trend data
    # If var is sea ice
    if var == 'aice':
        # Load 20-year trends
        ds_mon_trd = LoadData('Linear.Trend.20yr', var, 'month')
        ds_20trd = ds_mon_trd.loc[dict(month=type_onemonth)]

        # Load 40-year trends
        ds_mon_trd = LoadData('Linear.Trend.40yr', var, 'month')
        ds_40trd = ds_mon_trd.loc[dict(month=type_onemonth)]

    # Else var is temperature or AMOC
    else:
        # Load 20-year trends
        ds_20trd = LoadData('Linear.Trend.20yr', var, 'year')

        # Load 40-year trends
        ds_40trd = LoadData('Linear.Trend.40yr', var, 'year')

    ds_20trd = AddMidYear(ds_20trd)
    ds_40trd = AddMidYear(ds_40trd)

    if var == 'MOC' and o_domain and not ens_type:
        ds_abs = ds_abs.sel(lat=ocn_lat, method='nearest')
        ds_20trd = ds_20trd.sel(lat=ocn_lat, method='nearest')
        ds_40trd = ds_40trd.sel(lat=ocn_lat, method='nearest')
    elif var == 'MOC' and o_domain and ens_type:
        ds_abs_def = xr.open_dataset(path_to_data+'Linear.abs.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')
        ds_20trd_def = xr.open_dataset(path_to_data+'Linear.Trend.20yr.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')
        ds_40trd_def = xr.open_dataset(path_to_data+'Linear.Trend.40yr.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')

        ds_20trd_def = AddMidYear(ds_20trd_def)
        ds_40trd_def = AddMidYear(ds_40trd_def)

Linear.abs.aice.Arctic.1950.Sep.All_members.nc
Linear.Trend.20yr.aice.Arctic.1950.month.All_members.nc
Linear.Trend.40yr.aice.Arctic.1950.month.All_members.nc
CPU times: user 34.3 ms, sys: 3.8 ms, total: 38.1 ms
Wall time: 85.5 ms


### Make plots

In [52]:
%%time

if plots['comb'][0]:
    obs_list = [dsname for dsname, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]
    
    row_dict = {0: ['LENS2 piControl','PiC_UVnudge_LM2006']+obs_list,
                1: ['GHG2', 'GHG_UVnudge_LM']+obs_list,
                2: ['LENS2 HIST-SSP370', 'HIST_UVnudge_LM']+obs_list}
                # List: [0:dataset, 1:x dimension, 2:trend length, 3:y-label, 4:y-limit, 5:x-label, 6:x-limit]
    col_dict = {0: [ds_abs, period, 1, ylabelabs[var], ylimabs[var], xlabelabs, xlimabs],
                1: [ds_20trd, xdim, 20, ylabeltwolinesingle[var], ylimtrd20[var], xlabeltrd(20), xlimtrd(20)],
                2: [ds_40trd, xdim, 40, ylabeltwolinesingle[var], ylimtrd40[var], xlabeltrd(40), xlimtrd(40)]}
    
    # Set up figure and subplots
    fig = plt.figure(figsize=(11,8),layout='constrained')
    gs = gridspec.GridSpec(hgt+1, wdth, height_ratios=[1,1,1,0.3],
                            width_ratios=[ywidth(1),ywidth(20),ywidth(40)],
                            wspace=0.4, hspace=0.6)
    hand_list = []
    label_list = []
    let_labels = FigLabelList(hgt*wdth)
    
    # Plot all data
    for r, dnames in row_dict.items():
        # Create axes for given row
        axa = fig.add_subplot(gs[r,0])
        ax20 = fig.add_subplot(gs[r,1])
        ax40 = fig.add_subplot(gs[r,2])
        axlist = [axa, ax20, ax40]

        for c, clist in col_dict.items():
            ax = axlist[c]
            ds = clist[0]
            if c != 0:
                ax.axhline(0, color='k', zorder=0, lw=0.5)
                
            for dsname in dnames:
                attrs = ds_names[dsname]
                not_obs = dsname in ['GISTEMP','HadCRUT5','BEST']
                
                 # If LENS ensemble
                if attrs[1] == 0:
                    # Extract mean & min and max of envelope
                    da_mean = ds[dsname+' mean']
                    da_std = ds[dsname].std('slice')*2
                    da_min = da_mean-da_std
                    da_max = da_mean+da_std
                
                    # Plot envelope
                    ax.fill_between(da_min[clist[1]].values, da_min.values,
                        da_max.values, color=attrs[2], alpha=0.1,
                        ec=None,label=ds_paper_names[dsname], zorder=attrs[4])
                    
                    # Plot mean
                    da_mean.plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4],label=ds_paper_names[dsname]+' mean')

                # If nudging ensemble
                elif attrs[1] == 1:
                    da = ds[dsname]
                    index = dict(ensemble_member=1)

                    # Plot first ensemble member
                    da.loc[index].plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4], label=ds_paper_names[dsname]) 
    
                    # Plot other ensemble members
                    for i in range(2,4):
                        index['ensemble_member'] = i
                
                        da.loc[index].plot(
                            ax=ax, color=attrs[2], x=clist[1], zorder=attrs[4], ls=attrs[3])  
                        
                # If observations
                elif attrs[1] == 3:
                    if not ((c == 0 and not_obs) or (c == 2 and var == 'MOC')):
                        if (dsname == 'NSIDC' or dsname == 'OSISAF'):
                            t_slice = dict(time=slice('1980-01-01','2024-12-31')) if c == 0 else dict(start_year=slice(1980,2005))
                            da = ds[dsname].loc[t_slice]
                        elif dsname == 'RAPID':
                            t_slice = dict(year=slice(2004,2024)) if c == 0 else dict(start_year=slice(2004,2005))
                            da = ds[dsname].loc[t_slice]
                        else:
                            da = ds[dsname]
                        da.plot(
                            ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4], label=ds_paper_names[dsname])

            # Formatting
            ax.set_title('')
            ax.set_ylabel(clist[3])
            ax.set_ylim(clist[4])
            p = 3*r+c
            ax.annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontweight='bold')
            if var == 'aice' and c == 0:
                ax.xaxis.set_major_locator(mdates.YearLocator(10,month=type_onemonth))
                ax.xaxis.set_minor_locator(mdates.YearLocator(5,month=type_onemonth))
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
                ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),
                            np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
            else:
                ax.set_xticks(ticks=xticks_loc, minor=False)
                ax.set_xticks(ticks=xticks_locmin, minor=True)
                ax.set_xlim(clist[6])
            ax.set_xlabel(clist[5])
            ax.grid(alpha=0.3,which='both')
            if c == 0:
                ax.set_yticks(ticks=yticksabs[var])

            if c == 1:
                # Legend data collection
                if r == 0:
                    h, l = ax.get_legend_handles_labels()
                    print(l)
                    hand_list += h[3:]
                    label_list += l[3:]
                    hand_list += h[0:3]
                    label_list += l[0:3]
                elif r != 0:
                    h, l = ax.get_legend_handles_labels()
                    hand_list += h[0:3]
                    label_list += l[0:3]

    # Create legend
    axleg = fig.add_subplot(gs[3,:])
    axleg.legend(hand_list, label_list, loc='upper center', ncol=4, frameon=False)
    axleg.axis('off')
    
    SaveFig(fig, graph_type_str, date_str, var, note)

    plt.close(fig)

['PI-control', 'PI-control mean', 'PI+winds', 'OBS (ERA5)', 'OBS (NSIDC)', 'OBS (OSISAF)']
CPU times: user 998 ms, sys: 11.2 ms, total: 1.01 s
Wall time: 1.01 s


In [53]:
%%time

if plots['comb'][0] and o_domain and var == 'MOC' and ens_type:

    lat_sel = [20, 26.5, 33, 40, 45, 55, slice(33,55)]
    
    row_dict = {0: ['PiC_UVnudge_LM2006', 'Blues'],
                1: ['GHG_UVnudge_LM', 'Greens'],
                2: ['HIST_UVnudge_LM', 'Reds']}
                # List: [0:dataset, 1:x dimension, 2:trend length, 3:y-label, 4:y-limit, 5:x-label, 6:x-limit]
    col_dict = {0: [[ds_abs, ds_abs_def], period, 1, ylabelabs[var], ylimabs[var], xlabelabs, xlimabs],
                1: [[ds_20trd, ds_20trd_def], xdim, 20, ylabeltwolinesingle[var], ylimtrd20[var], xlabeltrd(20), xlimtrd(20)],
                2: [[ds_40trd, ds_40trd_def], xdim, 40, ylabeltwolinesingle[var], ylimtrd40[var], xlabeltrd(40), xlimtrd(40)]}
    
    # Set up figure and subplots
    fig = plt.figure(figsize=(11,8),layout='constrained')
    gs = gridspec.GridSpec(hgt+1, wdth, height_ratios=[1,1,1,0.3],
                            width_ratios=[ywidth(1),ywidth(20),ywidth(40)],
                            wspace=0.4, hspace=0.6)
    hand_list = []
    label_list = []
    let_labels = FigLabelList(hgt*wdth)
    
    # Plot all data
    for r, dnames in row_dict.items():
        # Create axes for given row
        axa = fig.add_subplot(gs[r,0])
        ax20 = fig.add_subplot(gs[r,1])
        ax40 = fig.add_subplot(gs[r,2])
        axlist = [axa, ax20, ax40]
        dsname = dnames[0]
        cmap = mpl.colormaps[dnames[1]]

        # Take colors at regular i
        colors = cmap(np.linspace(0.3, 1, len(lat_sel)))

        for c, clist in col_dict.items():
            ax = axlist[c]
            ds = clist[0][0][dnames[0]]
            ds_def = clist[0][1][dnames[0]]
            attrs = ds_names[dsname]
            if c != 0:
                ax.axhline(0, color='k', zorder=0, lw=0.5)
                
            for i in range(len(lat_sel)):
                if i == len(lat_sel)-1:
                    da = ds_def
                    addon = ' 33-55N'
                    lw = 2
                else:
                    da = ds.sel(lat=lat_sel[i], method='nearest')
                    addon = ' '+str(lat_sel[i])+'N'
                    lw = 1

                # Plot ensemble mean
                da.plot(
                    ax=ax, color=colors[i], ls=attrs[3], x=clist[1], zorder=attrs[4],
                    label=ds_paper_names[dsname]+addon, lw=lw) 

                        
            # Formatting
            ax.set_title('')
            ax.set_ylabel(clist[3])
            ax.set_ylim(clist[4])
            p = 3*r+c
            ax.annotate(let_labels[p], (0,1.05), xycoords='axes fraction')
            if var == 'aice' and c == 0:
                ax.xaxis.set_major_locator(mdates.YearLocator(10,month=type_onemonth))
                ax.xaxis.set_minor_locator(mdates.YearLocator(5,month=type_onemonth))
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
                ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),
                            np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
            else:
                ax.set_xticks(ticks=xticks_loc, minor=False)
                ax.set_xticks(ticks=xticks_locmin, minor=True)
                ax.set_xlim(clist[6])
            ax.set_xlabel(clist[5])
            ax.grid(alpha=0.3,which='both')
            if c == 0:
                ax.set_yticks(ticks=yticksabs[var])

        # Legend data collection
        h, l = ax.get_legend_handles_labels()
        hand_list += h[:]
        label_list += l[:]

    # Create legend
    axleg = fig.add_subplot(gs[3,:])
    axleg.legend(hand_list, label_list, loc='upper center', ncol=3, frameon=False)
    axleg.axis('off')
    
    SaveFig(fig, graph_type_str, date_str, var, note)

    plt.close(fig)

CPU times: user 26 µs, sys: 0 ns, total: 26 µs
Wall time: 8.82 µs


## Spatial trend plots

### Set up

In [54]:
if plot_types['spatial']:
    if s_domain == 1:
        proj = ccrs.NorthPolarStereo()
        extent = [-180, 180, 50, 90]
    elif s_domain == 2:
        proj = ccrs.NorthPolarStereo()
        extent = [-180, 180, 20, 50]

    # Monthly
    if time_avg == 0:
        trendi = [i-1 for i in trend_months]
        date_str = mon_str[trendi]
        period = 'time'
        wdth = len(trend_months)
        spt_outstr = '-'.join(date_str)

    # Seasonally
    elif time_avg == 2:
        date_str = seas_str
        period = 'time'
        wdth = 4
        spt_outstr = time_outstr
    
    # Latitude, longitude, and level names (ERA5 and the CESM produced datasets have different names)
    xname = defaultdict(lambda: 'lon')
    xname['ERA5'] = 'lonE'
    yname = defaultdict(lambda: 'lat')
    yname['ERA5'] = 'latE'
    zname = defaultdict(lambda: 'plev')

    # Determine width of plots
    hgt = int(varnum/2) if var == 'PSL' else varnum

    # Plot letter
    plot_let_full = np.array(['(a)', '(b)', '(c)', '(d)',
                              '(e)', '(f)', '(g)', '(h)',
                              '(i)', '(j)', '(k)', '(l)',
                              '(m)', '(n)', '(o)', '(p)'])
    plot_let = plot_let_full[0:(wdth*hgt)].reshape((hgt, wdth))

    # Do we add land features
    addland = (comp == 'ice')

In [55]:
if plots['strd'][0]:
    graph_type_str = 'Map.Trend'

    if time_avg == 0:
        index_list = date_str
    elif time_avg == 2:
        index_list = date_str

    sbpt_shp = (hgt,wdth)
    figsz = (2*wdth,2*hgt)

    # Plot levels
    aice_levels = np.arange(-30.0, 31.0, 2.0)
    aice_levels[15] = 0.00001
    hi_levels = np.arange(-1.6,1.61,0.1)
    hi_levels[0] = -5
    hi_levels[-1] = 5
    fsntoa_levels = np.array([-30,-25,-20,-15,-10,-7.5,-5,-2.5,-1,-0.5,0.0,0.5,1,2.5,5,7.5,10,15,20,25,30])
    aerod_levels = np.array([-0.07,-0.05,-0.03,-0.01,-0.009,-0.008,-0.007,-0.006,-0.005,-0.004,-0.003,-0.002,-0.001,0.0,
                            0.001,0.002,0.003,0.004,0.005,0.006,0.007,0.008,0.009,0.01,0.03,0.05,0.07])
    trdlevels = {'TREFHT': np.arange(-2.6,2.7,0.2),
                  'aice': aice_levels,
                  'hi': hi_levels,
                  'Z3': np.arange(-40,41,2),
                  'U': np.arange(-1.5,1.6,0.1),
                  'V': np.arange(-1.50,1.51,0.1),
                  'FSNTOA': fsntoa_levels,
                  'TGCLDLWP': fsntoa_levels,
                  'AEROD_v': aerod_levels}
    
    trdticks = {'TREFHT': np.arange(-2.5,2.6,0.5),
                'aice': np.arange(-30,31,10),
                'hi': np.arange(-1.6,1.7,0.2),
                'Z3': np.arange(-40,41,10),
                'U': np.array([-1.5,-1.0,-0.5,0.0,0.5,1.0,1.5]),
                'V': np.arange(-1.50,1.51,0.5),
                'FSNTOA': np.array([-30,-20,-10,-7.5,-5,-2.5,-1,-0.5,0.0,0.5,1,2.5,5,7.5,10,20,30]),
                'TGCLDLWP': np.array([-30,-20,-10,-7.5,-5,-2.5,-1,-0.5,0.0,0.5,1,2.5,5,7.5,10,20,30]),
                'AEROD_v': np.array([-0.07,-0.05,-0.03,-0.01,-0.005,0.0,0.005,0.01,0.03,0.05,0.07])}
    trdticks['hi'][0] = -5
    trdticks['hi'][-1] = 5
    
    # Plotting variables
    trdcmaps = {'TREFHT': cm.vik,
                'aice': SubCmap(mpl.colormaps['seismic_r'], trdlevels['aice'], 'mid', np.array([0,0,0,0])),
                'hi': SubCmap(mpl.colormaps['seismic_r'], trdlevels['hi'], 'mid', np.array([0,0,0,0])),
                'Z3': SubCmap(cm.vik, trdlevels['Z3'], 'mid', np.array([0,0,0,0])), 
                'U': SubCmap(cm.vik, trdlevels['U'], 'mid', np.array([0,0,0,0])),
                'V': SubCmap(cm.vik, trdlevels['V'], 'mid', np.array([0,0,0,0])),
                'FSNTOA': cm.vik,
                'TGCLDLWP': cm.cork,
                'AEROD_v': cm.vik}
    trdlabels = {'TREFHT': '2m air temperature trend (K/decade)',
                 'aice': 'Sea ice concentration trend (%/decade)',
                 'hi': 'Sea ice thickness trend (m/decade)',
                 'Z3': 'Geopotential height trend (m/decade)',
                  'U': 'Zonal wind trend (m s$^{-1}$/decade)',
                  'V': 'Meridional wind trend (m s$^{-1}$/decade)',
                  'FSNTOA': 'Net TOA shortwave trend (W m$^{-2}$/decade)',
                  'TGCLDLWP': 'Liquid water path trend (g m$^{-2}$/decade)',
                  'AEROD_v': 'Aerosol optical depth trend (unit/decade)'}
    trdlabels1row = {'TREFHT': '2m air temperature\ntrend (K/decade)',
                 'aice': 'Sea ice concentration\ntrend (%/decade)',
                 'hi': 'Sea ice thickness\ntrend (m/decade)',
                 'Z3': 'Geopotential height\ntrend (m/decade)',
                  'U': 'Zonal wind\ntrend (m s$^{-1}$/decade)',
                  'V': 'Meridional wind\ntrend (m s$^{-1}$/decade)',
                  'FSNTOA': 'Net TOA shortwave\ntrend (W m$^{-2}$/decade)',
                  'TGCLDLWP': 'Liquid water path\ntrend (g m$^{-2}$/decade)',
                  'AEROD_v': 'Aerosol optical depth\ntrend (unit/decade)'}

### Data loading

In [56]:
%%time

if plots['strd'][0]:
    if s_domain == 1:
        # Load spatial trends
        ds_sptrend = LoadData(graph_type_str, var, time_outstr)
        ds_sptrend = ds_sptrend*100 if var == 'aice' else ds_sptrend
        ds_sptrend = ds_sptrend*1000 if var == 'TGCLDLWP' else ds_sptrend

        if use_era5:
            ds_sppcorr = LoadData(graph_type_str+'.pcorr', var, time_outstr)

    if var == 'Z3':
        ds_usptrend = LoadData(graph_type_str, 'U', time_outstr)

        ds_vsptrend = LoadData(graph_type_str, 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.2 µs


### Make plots

#### 2D plots

In [57]:
%%time

if plots['strd'][0] and file_bool:
    row_num = {'ERA5': 0,
               'PiC_UVnudge_LM2006': 1,
               'GHG_UVnudge_LM': 2,
               'HIST_UVnudge_LM': 3}
    if not use_era5:
        row_num.pop('ERA5')
        for n, r in row_num.items():
            row_num[n] = r-1;
    
    # Setup figure
    fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained', subplot_kw=dict(projection=proj))
    fig.set_size_inches(figsz[0],figsz[1])
    
    # Plotting of trends for 2m air temperature and sea ice
    for i in np.arange(0,len(index_list)):
        # Time index
        t_index = {time_outstr: index_list[i]}

        for dsname, attrs in ds_names.items():
            if attrs[0]:
                da_trend = ds_sptrend[dsname].loc[t_index]

                # Plot data
                cax = da_trend.plot.contourf(
                        ax=axlist[row_num[dsname],i], cmap=trdcmaps[var], x=xname[dsname], y=yname[dsname],
                        levels=trdlevels[var], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)

                # Formatting title & labels
                if use_era5 and 'UVnudge' in dsname:
                    da_corr = ds_sppcorr[dsname].loc[t_index]
                    addon = ' r = {:.2f}'.format(da_corr.values)
                else:
                    addon = ' '
                AxisLabels(axlist[row_num[dsname],i], plot_let[row_num[dsname], i]+addon, addland)
                
    
    # Add colorbar 
    cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.1,shrink=0.75,fraction=0.05, extend='both')
    cb.set_label(label=trdlabels[var], fontsize=12)
    cb.ax.tick_params(labelsize=12)
    cb.set_ticks(ticks=trdticks[var])

    # Add row & column labels
    row_labels = [ds_paper_names[dsname] for dsname, attrs in row_num.items()]
    for ax, row in zip(axlist[:,0],row_labels):
        ax.annotate(row,(0,0.5),xytext=(-10,0),ha='right',va='center',
                    size=12,rotation=90,xycoords='axes fraction',textcoords='offset points')

    column_labels = date_str
    for ax, col in zip(axlist[0,:], column_labels):
        ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')

    SaveFig(fig, graph_type_str, spt_outstr, var, note)

    plt.close(fig)

CPU times: user 5 µs, sys: 1 µs, total: 6 µs
Wall time: 7.15 µs


#### 3D plots

In [58]:
%%time

# 3D variable plots
if plots['strd'][0] and not file_bool:
    row_labels = [ds_paper_names[dsname] for dsname, attrs in ds_names.items() if attrs[0]]

    if s_domain == 1:
        # Combine variables into dict
        trend_dict = {'Z3': [ds_sptrend],
                      'U': [ds_usptrend],
                      'V': [ds_vsptrend]}
        row_ind = {'ERA5': 0,
                   'PiC_UVnudge_LM2006': 1,
                   'GHG_UVnudge_LM': 2,
                   'HIST_UVnudge_LM': 3}
    else:
        trend_dict = {'U': [ds_usptrend],
                      'V': [ds_vsptrend]}
        
    # Cycle through variables
    for v, ds_sets in trend_dict.items():
        dv_trend = ds_sets[0]
        
        # Cycle through plot levels
        for pl in plot_levels:
            dl_trend = dv_trend.loc[dict(plev=pl)]
            
            # Setup figure
            fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained', subplot_kw=dict(projection=proj))
            fig.set_size_inches(figsz[0],figsz[1])

            if varnum == 1:
                axlist = np.expand_dims(axlist, axis=0)
            
            # Plotting of trends for geopotential height, zonal wind, and meridional wind
            for i in np.arange(0,len(index_list)):
                # Time index
                t_index = {time_outstr: index_list[i]}

                if varnum == 1:
                   axes_loc = {'ERA5': axlist[0, i]} 
                else:
                    axes_loc = {'ERA5': axlist[0, i],
                                'PiC_UVnudge_LM2006': axlist[1,i],
                                'GHG_UVnudge_LM': axlist[2,i],
                                'HIST_UVnudge_LM': axlist[3,i]}
        
                for dsname, attrs in ds_names.items():
                    if attrs[0]:
                        da_trend = dl_trend[dsname].loc[t_index]
        
                        # Plot data
                        # x=xname[dsname], y=yname[dsname],
                        cax = da_trend.plot.contourf(
                                ax=axes_loc[dsname],  cmap=trdcmaps[v], 
                                levels=trdlevels[v], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)
        
                        # Formatting title & labels
                        AxisLabels(axes_loc[dsname], plot_let[row_ind[dsname], i], addland)
                        

            if varnum == 1:
                cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.1,shrink=0.9,fraction=0.15, extend='both')
                cb.set_label(label=trdlabels1row[v], fontsize=12)
                cb.ax.tick_params(labelsize=12)
                cb.set_ticks(ticks=trdticks[v])
               
            else:
                cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.1,shrink=0.75,fraction=0.1, extend='both')
                cb.set_label(label=trdlabels[v], fontsize=12)
                cb.ax.tick_params(labelsize=12)
                cb.set_ticks(ticks=trdticks[v])

                # Add column labels
                column_labels = date_str
                for ax, col in zip(axlist[0,:], column_labels):
                    ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                                size=12,xycoords='axes fraction',textcoords='offset points')
                
                # Add row labels
                for ax, row in zip(axlist[:,0],row_labels):
                    ax.annotate(row,(0,0.5),xytext=(-10,0),ha='right',va='center',
                                size=12,rotation=90,xycoords='axes fraction',textcoords='offset points')
        
            SaveFig(fig, graph_type_str, time_outstr, v, note, pl)
        
            plt.close(fig)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.39 µs


#### 3D plots - U/V only

In [59]:
%%time

# 3D variable plots - combine U&V into single plot
if plots['strd'][0] and not file_bool:
    row_labels = [ds_paper_names[dsname] for dsname, attrs in ds_names.items() if attrs[0]]

    plot_let = plot_let_full[0:(wdth*hgt*2)].reshape((hgt*2, wdth))

    if s_domain == 1:
        # Combine variables into dict
        trend_dict = {'U': [ds_usptrend, 0],
                      'V': [ds_vsptrend, 1]}
        
    else:
        trend_dict = {'U': [ds_usptrend],
                      'V': [ds_vsptrend]}
     

    # Cycle through plot levels
    for pl in plot_levels:
        # Setup figure
        fig, axlist = plt.subplots(sbpt_shp[0]*2, sbpt_shp[1], layout='constrained', subplot_kw=dict(projection=proj))
        fig.set_size_inches(figsz[0],figsz[1]*2)
        
        
        # Cycle through variables
        for v, ds_sets in trend_dict.items():
            dl_trend = ds_sets[0].loc[dict(plev=pl)]
            axi = ds_sets[2]
            
            # Plotting of trends for geopotential height, zonal wind, and meridional wind
            for i in np.arange(0,len(index_list)):
                # Time index
                t_index = {time_outstr: index_list[i]}

                if varnum == 1:
                   axes_loc = {'ERA5': axlist[axi, i]} 
                else:
                    axes_loc = {'ERA5': axlist[0, i],
                                'PiC_UVnudge_LM2006': axlist[1,i],
                                'GHG_UVnudge_LM': axlist[1,i],
                                'HIST_UVnudge_LM': axlist[1,i]}
        
                for dsname, attrs in ds_names.items():
                    if attrs[0]:
                        da_trend = dl_trend[dsname].loc[t_index]
                        
                        # Plot data
                        # x=xname[dsname], y=yname[dsname],
                        cax = da_trend.plot.contourf(
                                ax=axes_loc[dsname],  cmap=trdcmaps[v], 
                                levels=trdlevels[v], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)
        
                        # Formatting title & labels
                        addon = ''
                        if 'UVnudge' in dsname:
                            AxisLabels(axes_loc[dsname], plot_let[1, i]+addon, addland)
                        else:
                            AxisLabels(axes_loc[dsname], plot_let[axi, i]+addon, addland)
                        

            

            if varnum == 1:
                cb = fig.colorbar(cax, ax=axlist[axi,:], pad=0.1,shrink=0.9,fraction=0.15, extend='both')
                cb.set_label(label=trdlabels1row[v], fontsize=12)
                cb.ax.tick_params(labelsize=12)
                cb.set_ticks(ticks=trdticks[v])
               
            else:
                cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.1,shrink=0.75,fraction=0.1, extend='both')
                cb.set_label(label=trdlabels[v], fontsize=12)
                cb.ax.tick_params(labelsize=12)
                cb.set_ticks(ticks=trdticks[v])

                # Add row labels
                for ax, row in zip(axlist[:,0],row_labels):
                    ax.annotate(row,(0,0.5),xytext=(-10,0),ha='right',va='center',
                                size=12,rotation=90,xycoords='axes fraction',textcoords='offset points')

            # Add column labels
            column_labels = date_str
            for ax, col in zip(axlist[0,:], column_labels):
                ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                            size=12,xycoords='axes fraction',textcoords='offset points')
                
                
        if varnum == 1:
            SaveFig(fig, graph_type_str, time_outstr, 'U-V', note, pl)
        else: 
            SaveFig(fig, graph_type_str, time_outstr, v, note, pl)
        
        plt.close(fig)

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 6.68 µs


## Spatial plots
### Set up

In [60]:
if plots['map'][0]:
    graph_type_str = 'Map'
        
    figsz = (3.5*wdth,3.5*hgt)
    sbpt_shp = (hgt,wdth)

    if time_avg == 0:
        index_list = date_str
    elif time_avg == 2:
        index_list = date_str

    # Plotting variables
    mcmaps = {'Z3': {'avg': cm.cork,
                     'var': cm.acton_r},
              'TREFHT': plt.cm.gist_rainbow_r,
              'aice': cm.devon,
              'hi': cm.devon,
              'FSNTOA': cm.lajolla,
              'TGCLDLWP': cm.navia,
              'AEROD_v': cm.lipari_r,
              'PSL': cm.batlow}
    mlabels = {'Z3': {'avg':' (m)', 'var': ' variance (m$^2$)'},
               'TREFHT': '2m air temperature (K)',
               'aice': 'Sea ice concetration',
               'hi': 'Sea ice thickness (m)',
               'FSNTOA': 'Net TOA shortwave (W m$^{-2}$)',
               'TGCLDLWP': 'Liquid water path (g m$^{-2}$)',
               'AEROD_v': 'Aerosol optical depth',
               'PSL': 'Sea level pressure (hPa)'}
    mlevels = {'Z3': {'avg':
                      {500: np.arange(-300,329,30), 
                      850: np.arange(-200,219,20), 
                      925: np.arange(-200,291,20)},
                      'var': 
                      {500: np.arange(0,3010,150), 
                      850: np.arange(0,3010,150), 
                      925: np.arange(0,3010,150)}
                     },   # Add to mean height
               'TREFHT': np.arange(220,301,2),
               'aice': np.array([0.0,0.05,0.10,0.15,0.20,0.30,0.40,0.50,0.60,0.70,0.80,0.90,0.95,1.00]),
               'hi': np.array([0.0,0.01,0.10,0.50,1.0,1.5,2.0,2.5,3.0,4.0,5.0]),
               'FSNTOA': np.arange(0,401,10),
               'TGCLDLWP': np.arange(0,401,10),
               'AEROD_v': np.arange(0,0.36,0.01),
               'PSL': np.arange(992,1038,1)}
    mheight = {500: 5300,
               850: 1400,
               925: 700}
    diffmlevels = {'TREFHT': np.arange(-20,21,2),
                   'aice': np.arange(-1,1.1,0.1),
                   'hi': np.array([-30,-5,-4.5,-4,-3.5,-3,-2.5,-2,-1.5,-1,-0.75,-0.5,-0.25,0,
                                   0.25,0.5,0.75,1,1.5,2,2.5,3,3.5,4,4.5,5,30]),
                   'FSNTOA': np.arange(-100,101,10),
                   'TGCLDLWP': np.array([-200,-150,-100,-90,-80,-70,-60,-50,-40,-30,-20,-10,0,
                                         10,20,30,40,50,60,70,80,90,100,150,200]),
                   'AEROD_v': np.arange(-0.1,0.11,0.01),
                   'PSL': np.arange(-10,10.4,1)}
    diffmticks = {'PSL': np.arange(-10,10.1,2)}
    diffmplotlabels = {'PSL': np.arange(-9,9.1,3)}
    diffmcmaps = {'TREFHT': cm.vik,
                  'aice': SubCmap(mpl.colormaps['seismic_r'], diffmlevels['aice'], 'mid', np.array([0,0,0,0])),
                  'hi': SubCmap(mpl.colormaps['seismic_r'], diffmlevels['hi'], 'mid', np.array([0,0,0,0])),
                  'FSNTOA': cm.vik,
                  'TGCLDLWP': cm.cork,
                  'AEROD_v': cm.vik,
                  'PSL': cm.bam}
    diffmlabels = {'TREFHT': 'Temperature difference (K)',
                   'aice': 'Sea ice concentration difference',
                   'hi': 'Sea ice thickness difference (m)',
                   'FSNTOA': 'Net TOA shortwave difference (W m$^{-2}$)',
                   'TGCLDLWP': 'Liquid water path difference (g m$^{-2}$)',
                   'AEROD_v': 'Aerosol optical depth difference',
                   'PSL': 'Sea level pressure difference (hPa)'}

### Data loading

In [61]:
%%time

if plots['map'][0]:
    # Load spatial averages
    ds_sp = LoadData(graph_type_str, var, time_outstr)
    ds_sp = ds_sp*1000 if var == 'TGCLDLWP' else ds_sp
    ds_sp = ds_sp/100 if var == 'PSL' else ds_sp
    

    if use_era5:
        regridderERA = LoadRegridder()
        ds_era = RegridERA5(ds_sp['ERA5'], regridderERA)

    if var == 'Z3':
        ds_spv = LoadData(graph_type_str+'.var', var, time_outstr)
        ds_usp = LoadData(graph_type_str, 'U', time_outstr)
        ds_vsp = LoadData(graph_type_str, 'V', time_outstr)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


### Make plots

In [62]:
%%time

if plots['map'][0] and file_bool:
    # Plotting of spatial patterns for 2m air temperature and sea ice
    row_num = {'ERA5': 0,
               'PiC_UVnudge_LM2006': 1,
               'GHG_UVnudge_LM': 2,
               'HIST_UVnudge_LM': 3}
    if not ds_names['ERA5'][0]:
        row_num.pop('ERA5')
        for n, r in row_num.items():
            row_num[n] = r-1;

    # Setup figure
    fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained', subplot_kw=dict(projection=proj))
    fig.set_size_inches(figsz[0],figsz[1])
    
    # Plotting of trends for 2m air temperature and sea ice
    for i in np.arange(0,len(index_list)):
        # Time index
        t_index = {time_outstr: index_list[i]}

        for dsname, attrs in ds_names.items():
            if attrs[0]:
                da_sp = ds_sp[dsname].loc[t_index]

                # Plot data
                cax = da_sp.plot.contourf(
                        ax=axlist[row_num[dsname],i], x=xname[dsname], y=yname[dsname], cmap=mcmaps[var], 
                        levels=mlevels[var], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)

                # Formatting
                addon = ''
                AxisLabels(axlist[row_num[dsname],i], plot_let[row_num[dsname], i]+addon, addland)

    # Add colorbar
    cb = fig.colorbar(cax,ax=axlist[:,:],pad=0.1,shrink=0.75,fraction=0.1, extend='both')
    cb.set_label(label=mlabels[var], fontsize=12)
    cb.ax.tick_params(labelsize=12)

    # Add row & column labels
    row_labels = [ds_paper_names[dsname] for dsname, attrs in row_num.items()]
    for ax, row in zip(axlist[:,0],row_labels):
        ax.annotate(row,(0,0.5),xytext=(-10,0),ha='right',va='center',
                    size=12,rotation=90,xycoords='axes fraction',textcoords='offset points')

    column_labels = date_str
    for ax, col in zip(axlist[0,:], column_labels):
        ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')

    SaveFig(fig, graph_type_str, spt_outstr, var, note)

    plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 7.15 µs


In [63]:
%%time

if plots['map'][0] and file_bool:
    # Plotting of spatial patterns for 2m air temperature and sea ice - difference
    row_num = {'ERA5': 0,
               'PiC_UVnudge_LM2006': 1,
               'GHG_UVnudge_LM': 2,
               'HIST_UVnudge_LM': 3}
    if not use_era5:
        row_num.pop('ERA5')
        for n, r in row_num.items():
            row_num[n] = r-1
        row_num['HIST_UVnudge_LM'] = 0
        row_num['PiC_UVnudge_LM2006'] = 2
    
    # Setup figure
    fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained', subplot_kw=dict(projection=proj))
    fig.set_size_inches(figsz[0],figsz[1])

    tot_name = 'ERA5' if use_era5 else 'HIST_UVnudge_LM'
    
    # Plotting of trends for 2m air temperature and sea ice
    for i in np.arange(0,len(index_list)):
        # Time index
        t_index = {time_outstr: index_list[i]}
        da_tot = ds_sp[tot_name].loc[t_index] if tot_name != 'ERA5' else ds_era.loc[t_index]

        for dsname, attrs in ds_names.items():
            if attrs[0] and dsname != tot_name:
                da_sp = ds_sp[dsname].loc[t_index]
                da_sp = da_sp.assign_coords({'lat':da_tot.lat})

                # Plot data
                dcax = (da_tot-da_sp).plot.contourf(
                        ax=axlist[row_num[dsname],i], x=xname[dsname], y=yname[dsname], cmap=diffmcmaps[var], 
                        levels=diffmlevels[var], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)

                # Formatting
                addon = ''
                AxisLabels(axlist[row_num[dsname],i], plot_let[row_num[dsname], i]+addon, addland)
                
            elif dsname == tot_name:
                da_sp = ds_sp[dsname].loc[t_index]
                
                # Plot data
                cax = da_sp.plot.contourf(
                        ax=axlist[row_num[dsname],i], x=xname[dsname], y=yname[dsname], cmap=mcmaps[var], 
                        levels=mlevels[var], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)

                # Formatting
                addon = ''
                AxisLabels(axlist[row_num[dsname],i], plot_let[row_num[dsname], i]+addon, addland)

    # Add colorbar
    cb = fig.colorbar(cax,ax=axlist[:,:],pad=0.1,shrink=1.25,fraction=0.1, extend='both')
    cb.set_label(label=mlabels[var], fontsize=12)
    cb.ax.tick_params(labelsize=12)

    dcb = fig.colorbar(dcax,ax=axlist[:,:],pad=0.1,shrink=1.25,fraction=0.1, extend='both')
    dcb.set_label(label=diffmlabels[var], fontsize=12)
    dcb.ax.tick_params(labelsize=12)

    # Add row & column labels
    row_labels = [ds_paper_names[dsname] for dsname, attrs in row_num.items()]
    row_labels = row_labels if use_era5 else list(reversed(row_labels))
    for ax, row in zip(axlist[:,0],row_labels):
        ax.annotate(row,(0,0.5),xytext=(-10,0),ha='right',va='center',
                    size=12,rotation=90,xycoords='axes fraction',textcoords='offset points')

    column_labels = date_str
    for ax, col in zip(axlist[0,:], column_labels):
        ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')

    SaveFig(fig, graph_type_str+'.diff', spt_outstr, var, note)

    plt.close(fig)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.63 µs


In [64]:
def fmt(x):
    return str(int(x))+' hPa'

In [65]:
%%time

if plots['map'][0] and file_bool:
    # Plotting of spatial patterns for difference between nudged & freely-evolving

    row_vals = {0:['PiC_UVnudge_LM2006', 'LENS2 piControl'],
                1:['GHG_UVnudge_LM', 'GHG2'],
                2:['HIST_UVnudge_LM', 'LENS2 HIST-SSP370']}
    row_labels = []

    # Setup figure
    wr = np.ones(sbpt_shp[1]+1)
    wr[-1] = 0.1
    fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1]+1, layout='constrained', width_ratios=wr, subplot_kw=dict(projection=proj))
    fig.set_size_inches(figsz[0],figsz[1])
    gs = axlist[0,4].get_gridspec()
    for ax in axlist[:,4]:
        ax.remove()
    cbar_ax = fig.add_subplot(gs[:,4])
    
    # Plotting of trends for 2m air temperature and sea ice
    for r, nlist in row_vals.items():
        ds_diff = ds_sp[nlist[0]]-ds_sp[nlist[1]+' mean']
        row_labels.append(ds_paper_names[nlist[0]]+'$-$\n'+ds_paper_names[nlist[1]])

        for i in np.arange(0,len(index_list)):
            # Time index
            t_index = {time_outstr: index_list[i]}
            da_diff = ds_diff.loc[t_index]

            # Plot data
            dcax = da_diff.plot.contourf(
                    ax=axlist[r,i], x=xname[nlist[0]], y=yname[nlist[0]], cmap=diffmcmaps[var], 
                    levels=diffmlevels[var], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=1)
            # dlab = da_diff.plot.contour(
            #         ax=axlist[r,i], x=xname[nlist[0]], y=yname[nlist[0]], colors='k', 
            #         levels=[0], add_colorbar=False,transform=ccrs.PlateCarree(),zorder=10)

            # Formatting
            addon = ''
            AxisLabels(axlist[r,i], addon, addland)
            # axlist[r,i].clabel(dlab, dlab.levels, fmt=fmt, fontsize=14)
                
    # Add difference color bar
    dcb = fig.colorbar(dcax,cax=cbar_ax, shrink=0.1, extend='both')
    dcb.set_label(label=diffmlabels[var], fontsize=16)
    dcb.ax.tick_params(labelsize=14)
    dcb.set_ticks(ticks=diffmticks[var])

    # Add row & column labels
    for ax, row in zip(axlist[:,0],row_labels):
        ax.annotate(row,(0,0.5),xytext=(-10,0),ha='center',va='center',
                    size=18,rotation=90,xycoords='axes fraction',textcoords='offset points')

    column_labels = date_str
    for ax, col in zip(axlist[0,:], column_labels):
        ax.annotate(col,(0.5,1),xytext=(0,10),ha='center',va='center',
                    size=18,xycoords='axes fraction',textcoords='offset points')

    SaveFig(fig, graph_type_str+'.diff', spt_outstr, var, note)

    plt.close(fig)

CPU times: user 6 µs, sys: 0 ns, total: 6 µs
Wall time: 7.87 µs


## Zonal trend plots
### Set up

In [66]:
if plot_types['zonal']:
    if s_domain == 1:
        xlim = [50, 90]
        latticks = np.arange(50,91,10)
    elif s_domain == 2:
        xlim = [20, 50]
        latticks = np.arange(20,51,10)

    # Monthly
    if time_avg == 0:
        date_str = mon_str
        period = 'time'
        index_list = date_str

    # Seasonally
    elif time_avg == 2:
        date_str = seas_str
        period = 'time'
        index_list = date_str
    
    # Latitude and level names (ERA5 and the CESM produced datasets have different names)
    yname = defaultdict(lambda: 'lat')
    yname['ERA5'] = 'latE'
    zname = defaultdict(lambda: 'plev')

    pticks = np.array([100,300,500,700,850,1000])
    plim = [1000, 100]

In [67]:
if plots['ztrd'][0]:
    graph_type_str = 'Zonal.Trend'

    # Determine width of plots
    wdth = 4
    hgt = 1 if varnum == 1 else 2

    sbpt_shp = (hgt,wdth)
    figsz = (2*wdth,2*hgt)

    # Plot letter
    plot_let_full = np.array(['(a)', '(b)', '(c)', '(d)',
                              '(e)', '(f)', '(g)', '(h)'])
    plot_let = plot_let_full[0:(wdth*hgt)].reshape((hgt, wdth))

    # Plot levels
    ztrdlevels = {'Z3': np.arange(-40,41,2),
                  'U': np.arange(-1.5,1.6,0.1),
                  'V': np.arange(-0.20,0.21,0.01)}

    ztrdcmaps = {'Z3': SubCmap(cm.vik, ztrdlevels['Z3'], 'mid', np.array([0,0,0,0])), 
                'U': SubCmap(cm.vik, ztrdlevels['U'], 'mid', np.array([0,0,0,0])),
                'V': SubCmap(cm.vik, ztrdlevels['V'], 'mid', np.array([0,0,0,0]))}

    ztrdticks = {'Z3': np.arange(-40,41,10),
                'U': np.array([-1.5,-1.0,-0.5,0.0,0.5,1.0,1.5]),
                'V': np.arange(-0.20,0.21,0.05)}

    # Plotting variables
    ztrdlabels = {'Z3': 'Geopotential height trend (m/decade)',
                  'U': 'Zonal wind trend (m s$^{-1}$/decade)',
                  'V': 'Meridional wind trend (m s$^{-1}$/decade)'}
    ztrdlabels1row = {'Z3': 'Geopotential height\ntrend (m/decade)',
                  'U': 'Zonal wind\ntrend (m s$^{-1}$/decade)',
                  'V': 'Meridional wind\ntrend (m s$^{-1}$/decade)'}

### Data loading

In [68]:
%%time

# Load zonal trends
if plots['ztrd'][0]:
    ds_uztrend = LoadData(graph_type_str, 'U', time_outstr)
    ds_vztrend = LoadData(graph_type_str, 'V', time_outstr)

    ds_uzpval = LoadData(graph_type_str+'.pval', 'U', time_outstr)
    ds_vzpval = LoadData(graph_type_str+'.pval', 'V', time_outstr)

    if s_domain == 1:
        ds_zpval = LoadData(graph_type_str+'.pval', var, time_outstr)
        ds_ztrend = LoadData(graph_type_str, var, time_outstr)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 7.15 µs


### Make plots

In [69]:
%%time

if plots['ztrd'][0]:
    

    if s_domain == 1:
        # List of variables
        vards = {'Z3': [ds_ztrend, ds_zpval],
                 'U': [ds_uztrend, ds_uzpval],
                 'V': [ds_vztrend, ds_vzpval]}
    else:
        vards = {'U': [ds_uztrend, ds_uzpval],
                 'V': [ds_vztrend, ds_vzpval]}

    for v, ds_sets in vards.items():
        dv_ztrd = ds_sets[0]
        dv_zpval = ds_sets[1]

        pval_names = list(dv_zpval.keys())
        
        # Setup figure
        fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained')
        fig.set_size_inches(figsz[0],figsz[1])

        if varnum == 1:
            axlist = np.expand_dims(axlist, axis=0)
        
        for i in np.arange(0,len(index_list)):
            # Time index
            t_index = {time_outstr: index_list[i]}

            # Plotting zonal U & V winds
            if varnum == 1:
                axes_loc = {'ERA5': axlist[0, i]}
            else:
                axes_loc = {'ERA5': axlist[0, i],
                            'PiC_UVnudge_2006': axlist[1,i],
                            'PiC_UVnudge_LM2006': axlist[1,i],
                            'PiC_UVnudge_MM2006': axlist[1,i]}

            for dsname, attrs in ds_names.items():
                if attrs[0] and dsname != 'LENS2 piControl':
                    da_ztrd = dv_ztrd[dsname].loc[t_index]

                    # Plot data
                    cax = da_ztrd.plot.contourf(
                            ax=axes_loc[dsname], x=yname[dsname], y=zname[dsname], cmap=ztrdcmaps[v], yincrease=False,
                            levels=ztrdlevels[v], add_colorbar=False, add_labels=False, zorder=1)

                    if dsname in pval_names:
                            da_pval = dv_zpval[dsname].loc[t_index]
                            da_crit = dv_zpval[dsname+' pcrit'].loc[t_index].values
                            da_pval.plot.contourf(
                                ax=axes_loc[dsname],x='lat', y=zname[dsname],levels=[-0.01,da_crit,1],hatches=['.',None],colors='none',
                                add_colorbar=False,add_labels=False,zorder=2)

                    if dsname == 'ERA5':
                        title_str = plot_let[0,i]+' '+date_str[i]
                        #title_str = plot_let[0,i]+date_str[i]+', '+ds_paper_names[dsname]
                    else: 
                        title_str = plot_let[1,i]+' '+date_str[i]+', '+ds_paper_names[dsname]

                    # Adding difference plot
                    if i == 0:
                        # Labeling
                        axes_loc[dsname].set_ylabel('Pressure level (hPa)')
                        axes_loc[dsname].set_yticks(pticks)
            
                    else:
                        # Labeling
                        axes_loc[dsname].set_ylabel('')
                        axes_loc[dsname].set_yticks(pticks, labels=[])

                    axes_loc[dsname].set_title(title_str, loc='left',fontsize=10)
                    axes_loc[dsname].set_ylim(plim)
                    axes_loc[dsname].set_xlabel('Latitude')
                    axes_loc[dsname].set_xticks(latticks)

        if varnum == 1:
            # Add colorbars 
            cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.05,fraction=0.2, extend='both')
            cb.set_label(label=ztrdlabels1row[v])
            #cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=ztrdticks[v])

        else:
            # Add colorbars 
            cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.05,shrink=0.75,fraction=0.1, extend='both')
            cb.set_label(label=ztrdlabels[v])
            #cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=ztrdticks[v])

        SaveFig(fig, graph_type_str, time_outstr, v, note)

        plt.close(fig)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 6.44 µs


In [70]:
%%time

# Put U & V on one plot
if plots['ztrd'][0]:
    

    if s_domain == 1:
        # List of variables
        vards = {'U': [ds_uztrend, ds_uzpval, 0],
                 'V': [ds_vztrend, ds_vzpval, 1]}
    else:
        vards = {'U': [ds_uztrend, ds_uzpval],
                 'V': [ds_vztrend, ds_vzpval]}

    plot_let = plot_let_full[0:(wdth*hgt*2)].reshape((hgt*2, wdth))
    
    # Setup figure
    fig, axlist = plt.subplots(sbpt_shp[0]*2, sbpt_shp[1], layout='constrained')
    fig.set_size_inches(figsz[0],figsz[1]*2)
    
    for v, ds_sets in vards.items():
        dv_ztrd = ds_sets[0]
        dv_zpval = ds_sets[1]
        axi = ds_sets[2]

        pval_names = list(dv_zpval.keys())
        
        for i in np.arange(0,len(index_list)):
            # Time index
            t_index = {time_outstr: index_list[i]}

            # Plotting zonal U & V winds
            if varnum == 1:
                axes_loc = {'ERA5': axlist[axi, i]}
            else:
                axes_loc = {'ERA5': axlist[0, i],
                            'PiC_UVnudge_2006': axlist[1,i],
                            'PiC_UVnudge_LM2006': axlist[1,i],
                            'PiC_UVnudge_MM2006': axlist[1,i]}

            for dsname, attrs in ds_names.items():
                if attrs[0] and dsname != 'LENS2 piControl':
                    da_ztrd = dv_ztrd[dsname].loc[t_index]

                    # Plot data
                    cax = da_ztrd.plot.contourf(
                            ax=axes_loc[dsname], x=yname[dsname], y=zname[dsname], cmap=ztrdcmaps[v], yincrease=False,
                            levels=ztrdlevels[v], add_colorbar=False, add_labels=False, zorder=1)

                    if dsname in pval_names:
                            da_pval = dv_zpval[dsname].loc[t_index]
                            da_crit = dv_zpval[dsname+' pcrit'].loc[t_index].values
                            da_pval.plot.contourf(
                                ax=axes_loc[dsname],x='lat', y=zname[dsname],levels=[-0.01,da_crit,1],hatches=['...',None],colors='none',
                                add_colorbar=False,add_labels=False,zorder=2)

                    if dsname == 'ERA5':
                        title_str = plot_let[axi,i]
                        #title_str = plot_let[0,i]+date_str[i]+', '+ds_paper_names[dsname]
                    else: 
                        title_str = plot_let[1,i]+' '+date_str[i]+', '+ds_paper_names[dsname]

                    # Adding difference plot
                    if i == 0:
                        # Labeling
                        axes_loc[dsname].set_ylabel('Pressure level (hPa)')
                        axes_loc[dsname].set_yticks(pticks)
            
                    else:
                        # Labeling
                        axes_loc[dsname].set_ylabel('')
                        axes_loc[dsname].set_yticks(pticks, labels=[])

                    if axi == 0:
                        axes_loc[dsname].set_xlabel('')
                    else: 
                        axes_loc[dsname].set_xlabel('Latitude')
                        
                    axes_loc[dsname].set_title(title_str, loc='left',fontsize=10)
                    axes_loc[dsname].set_ylim(plim)
                    
                    axes_loc[dsname].set_xticks(latticks)

        if varnum == 1:
            # Add colorbars 
            cb = fig.colorbar(cax, ax=axlist[axi,:], pad=0.05,fraction=0.2, extend='both')
            cb.set_label(label=ztrdlabels1row[v])
            #cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=ztrdticks[v])

        else:
            # Add colorbars 
            cb = fig.colorbar(cax, ax=axlist[:,:], pad=0.05,shrink=0.75,fraction=0.1, extend='both')
            cb.set_label(label=ztrdlabels[v])
            #cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=ztrdticks[v])

    # Add column labels
    column_labels = date_str
    for ax, col in zip(axlist[0,:], column_labels):
        ax.annotate(col,(0.5,1),xytext=(0,25),ha='center',va='center',
                    size=12,xycoords='axes fraction',textcoords='offset points')

    SaveFig(fig, graph_type_str, time_outstr, 'U-V', note)

    plt.close(fig)

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 6.68 µs


## Zonal plots
### Set up

In [71]:
if plots['zon'][0]:
    graph_type_str = 'Zonal'

    # Determine width of plots
    wdth = 3
    hgt = 1

    # Plot letter
    plot_let_full = np.array(['(a)', '(b)', '(c)', '(d)'])
    plot_let = plot_let_full[0:(wdth*hgt)].reshape((hgt, wdth))

    
    plot_let = {'ERA5': plot_let[0,0],
                'PiC_UVnudge_2006': plot_let[0,1],
                'PiC_UVnudge_LM2006': plot_let[0,1],
                'PiC_UVnudge_MM2006': plot_let[0,1],
                'diff': plot_let[0,2]}

    sbpt_shp = (hgt,wdth)
    figsz = (3*wdth,2*hgt)

    # Plot levels
    zonlevels = {'U': np.arange(-30,31,2),
                  'V': np.arange(-2.5,2.6,0.25)}
    
    zdifflevels = {'U': np.array([-5, -2, -1, -0.5, 0, 0.5, 1, 2, 5]),
                  'V': np.array([-5, -2, -1, -0.5, 0, 0.5, 1, 2, 5])}
    
    zoncmaps = {'U': cm.cork,
               'V': cm.bam}
    
    zdiffcmaps = {'U': SubCmap(mpl.colormaps['seismic_r'], zdifflevels['U'], 'mid', np.array([0,0,0,0])),
                  'V': SubCmap(mpl.colormaps['seismic_r'], zdifflevels['V'], 'mid', np.array([0,0,0,0]))}
    
    zonticks = {'U': np.array([-30, -20, -10, 0, 10, 20, 30]),
                  'V': np.array([-2, -1, 0, 1.0, 2.0])}
    
    zdiffticks = {'U': np.array([-5, -2, -1, -0.5, 0, 0.5, 1, 2, 5]),
                  'V': np.array([-5, -2, -1, -0.5, 0, 0.5, 1, 2, 5])}
    
    # Plotting variables
    zonlabels = {'U': 'Zonal wind (m s$^{-1}$)',
                  'V': 'Meridional wind (m s$^{-1}$)'}
    
    zdifflabels = {'U': 'Zonal wind error (m s$^{-1}$)',
                    'V': 'Meridional wind error (m s$^{-1}$)'}

### Data loading

In [72]:
%%time

# Load zonal averages
if plots['zon'][0]:
    ds_uzon = LoadData(graph_type_str, 'U', time_outstr)
    ds_vzon = LoadData(graph_type_str, 'V', time_outstr)

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 6.2 µs


### Make plots

In [73]:
%%time

if plots['zon'][0] and not file_bool:
    # List of wind variables
    varlist = np.array(['U', 'V'])
    for v in varlist:
        ds_zon = ds_uzon if v == 'U' else ds_vzon

        for i in np.arange(0,len(index_list)):
            # Time index
            t_index = {time_outstr: index_list[i]}

            # Setup figure
            fig, axlist = plt.subplots(sbpt_shp[0], sbpt_shp[1], layout='constrained')
            fig.set_size_inches(figsz[0],figsz[1])
        
            # Plotting zonal U & V winds
            axes_loc = {'ERA5': axlist[0],
                        'PiC_UVnudge_2006': axlist[1],
                        'PiC_UVnudge_LM2006': axlist[1],
                        'PiC_UVnudge_MM2006': axlist[1]}

            for dsname, attrs in ds_names.items():
                if attrs[0] and dsname != 'LENS2 piControl':
                    da_zon = ds_zon[dsname].loc[t_index]

                    # Plot data
                    cax = da_zon.plot.contourf(
                            ax=axes_loc[dsname], x=yname[dsname], y=zname[dsname], cmap=zoncmaps[v], yincrease=False,
                            levels=zonlevels[v], add_colorbar=False, add_labels=False, zorder=1)

                    # Adding difference plot
                    if 'PiC_UVnudge' in dsname:
                        da_era = ds_zon['ERA5'].loc[t_index]
                        da_era = da_era.rename({'latE':'lat', 'levE':'plev'})

                        cax2 = (da_zon-da_era).plot.contourf(
                                ax=axlist[2], x=yname[dsname], y=zname[dsname], cmap=zdiffcmaps[v], yincrease=False,
                                levels=zdifflevels[v], add_colorbar=False, add_labels=False, zorder=1)

                        axlist[2].fill([60,60,90,90],[100,850,850,100],
                            facecolor='gray',edgecolor='black',alpha=.4,linestyle='--',zorder=2)

                        # Labeling
                        axes_loc[dsname].set_title(plot_let[dsname]+' '+ds_paper_names[dsname], loc='left', fontsize=10)
                        axes_loc[dsname].set_ylabel('')
                        axes_loc[dsname].set_yticks(pticks, labels=[])
                        axes_loc[dsname].set_ylim(plim)
                        axes_loc[dsname].set_xlabel('Latitude')
                        axes_loc[dsname].set_xticks(latticks)

                        axlist[2].set_title(plot_let['diff']+' '+ds_paper_names[dsname]+'–'+'ERA5', loc='left', fontsize=10)
                        axlist[2].set_ylabel('')
                        axlist[2].set_yticks(pticks, labels=[])
                        axlist[2].set_ylim(plim)
                        axlist[2].set_xlabel('Latitude')
                        axlist[2].set_xticks(latticks)

                    else:
                        # Labeling
                        axes_loc[dsname].set_title(plot_let[dsname]+' '+ds_paper_names[dsname], loc='left', fontsize=10)
                        axes_loc[dsname].set_ylabel('Pressure level (hPa)')
                        axes_loc[dsname].set_yticks(pticks)
                        axes_loc[dsname].set_ylim(plim)
                        axes_loc[dsname].set_xlabel('Latitude')
                        axes_loc[dsname].set_xticks(latticks)
        
            # Add colorbars 
            cb = fig.colorbar(cax, ax=axlist[:], shrink=1.25, pad=0.05,fraction=0.1, extend='both')
            cb.set_label(label=zonlabels[v])
            #cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=zonticks[v])

            cb2 = fig.colorbar(cax2, ax=axlist[:], shrink=1.25, pad=0.05,fraction=0.1, extend='both')
            cb2.set_label(label=zdifflabels[v])
            #cb2.ax.tick_params(labelsize=12)
            cb2.set_ticks(ticks=zdiffticks[v])

            # SaveFig(fig, graph_type_str+'.noNW', date_str[i], v, note)
            SaveFig(fig, graph_type_str, date_str[i], v, note)

            plt.close(fig)

CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 7.39 µs
